# Hydraulisch-thermische Netzstudie: Großwärmespeicher MAN/UPM im Fernwärmenetz Stadtbach

**Stand:** Etappe 1 von 4 (Kapitel 1–8: Daten, Qualität, Heizkurve). Netzmodell, Optimierung, Szenarien und pandapipes folgen in den nächsten Etappen.

## 1. Management Summary
> *Wird am Ende des Workflows auf Basis der berechneten Ergebnisse erzeugt (Kapitel 22–23). Bis dahin gilt: keine belastbare Aussage zur Speicherwirkung.*

**Bisherige Zwischenbefunde (Etappe 1):** siehe Abschnitte 5.2 (Widersprüche), 6.3 (Datenqualität) und 8.4 (Heizkurve).

## 2. Zielsetzung und Systemgrenzen
**Frage:** Kann ein direkt eingebundener, druckbeaufschlagter Heißwasserspeicher bei MAN/UPM kritische Druck- und Temperatursituationen im bestehenden und ausgebauten Netz vermeiden oder reduzieren – und welche Speicher-, Pumpen-, Anschluss- und Betriebsparameter sind dafür nötig?

**Systemgrenze:** Fernwärmenetz Stadtbach (Netz Mitte/Ost/Süd/West) mit Erzeugern HKW, GT-Ost, BMHKW, AVA, Heizwerk Süd/West; Speicher ohne trennenden Wärmeübertrager.

**Grundsatz Speicherphysik:** Ein direkt eingebundener Speicher erzeugt *keine* Förderhöhe. Er kann Druckverluste nur über lokale Einspeisung und reduzierte Transportströme senken. Aktive Druckanhebung erfordert eine Pumpe. Diese Unterscheidung wird in Modell und Ergebnisdarstellung getrennt ausgewiesen.

**Arbeitsannahmen (mit dem Auftraggeber abgestimmt):** (i) Speicher als *Druckspeicher* wegen bis zu 120 °C im Winter; (ii) MAN-Anschlusspunkt wird als Annahme aus den Plänen festgelegt und als Sensitivität variiert; (iii) lineare Vorlauf-Heizkurve 120 °C bei −10 °C bis 90 °C bei +20 °C als Vorgabe.

## 3. Reproduzierbarkeit und Softwareumgebung

In [ ]:
import os, sys, re, json, math, platform, hashlib, warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# ---------------- zentrale Konfiguration ----------------
SEED = 20250101
np.random.seed(SEED)

def find_repo_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "calion").is_dir() and (p / "configs").is_dir():
            return p
    raise FileNotFoundError("Repository-Wurzel (Ordner 'calion' + 'configs') nicht gefunden.")

ROOT      = find_repo_root()
DATA_DIR  = ROOT / "data" / "Stadtbach" / "upload"      # NDA: nicht versionieren!
RESULTS   = ROOT / "results" / "studie_MAN_UPM"
FIG_DIR   = RESULTS / "abbildungen"
for d in (RESULTS, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

CFG = dict(
    year=2025,
    p_unit="bar (Überdruck, Annahme – siehe Annahmenkatalog A-01)",
    heizkurve_vorgabe=dict(T_low_out=-10.0, T_VL_at_low=120.0, T_high_out=20.0, T_VL_at_high=90.0),
    cp_kJ_kgK=4.19, rho_kg_m3=971.8,        # Näherung bei ~80 °C; später temperaturabhängig
    dt_h=1.0,
)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
COL = dict(vl="#c0392b", rl="#2874a6", ok="#1e8449", warn="#d68910", grau="#7f8c8d", mess="#34495e")

print("Python", platform.python_version(), "|", platform.platform())
for m in ("numpy","pandas","scipy","matplotlib","pyarrow","openpyxl","yaml","pyomo","pandapipes"):
    try:
        mod = __import__(m); print(f"{m:11s}", getattr(mod, "__version__", "?"))
    except Exception as e:
        print(f"{m:11s} NICHT installiert ({type(e).__name__})")
print("Repo-Wurzel:", ROOT.name, "| Datenordner vorhanden:", DATA_DIR.exists())

## 4. Repository- und Dateiinventur
Rekursive Bestandsaufnahme der für die Studie relevanten Verzeichnisse. Die Bedeutung wird – wo möglich – am **Inhalt** geprüft (YAML: Knoten/Leitungen; Parquet: Spalten/Zeilen; PDF: Seitenzahl), nicht am Dateinamen.
Schreibweise „Stadtbach": tolerante, case-insensitive Suche (der Auftrag nennt `stadtabcach`).

In [ ]:
import yaml
# --- 4.1 Tolerante Ordnersuche
def fuzzy_dirs(base, pattern=r"stadt.?a?b?a?c?h"):
    rx = re.compile(pattern, re.I)
    return sorted({str(p.relative_to(ROOT)) for p in base.rglob("*")
                   if p.is_dir() and rx.search(p.name) and ".git" not in p.parts})
print("Gefundene Stadtbach-Ordner:")
for d in fuzzy_dirs(ROOT): print("  ", d)

# --- 4.2 Inventur
SCAN = [ROOT/"configs", ROOT/"docs", ROOT/"paper2_package", ROOT/"scripts", ROOT/"tools", ROOT/"calion",
        ROOT/"revision", DATA_DIR]
EXT = {".yaml",".yml",".csv",".parquet",".json",".geojson",".shp",".dxf",".dwg",".pdf",".xlsx",".xls",
       ".ipynb",".md",".docx",".pptx",".py"}
rows=[]
for base in SCAN:
    if not base.exists(): continue
    for p in base.rglob("*"):
        if p.is_file() and p.suffix.lower() in EXT and ".git" not in p.parts and "_build" not in p.parts:
            rel = p.relative_to(ROOT).as_posix()
            if not re.search(r"stadt|WV6|640|acron|Import_Data|storage_geometry|pandapipes|Lastg", rel, re.I):
                continue
            st = p.stat()
            rows.append(dict(pfad=rel, typ=p.suffix.lower().lstrip("."), groesse_kB=round(st.st_size/1024,1),
                             geaendert=datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d")))
inv = pd.DataFrame(rows).sort_values("pfad").reset_index(drop=True)
print(len(inv), "relevante Dateien")

In [ ]:
# --- 4.3 Inhaltsprüfung + Bewertung
def inspect(rel):
    p = ROOT/rel; ext = p.suffix.lower()
    try:
        if ext in (".yaml",".yml"):
            d = yaml.safe_load(p.read_text(encoding="utf-8", errors="ignore"))
            n = d.get("network", {}) if isinstance(d, dict) else {}
            if n.get("nodes") and n.get("pipes"):
                return f"YAML-Netz: {len(n['nodes'])} Knoten, {len(n['pipes'])} Leitungen"
            return "YAML (kein Netz)"
        if ext == ".parquet":
            df = pd.read_parquet(p); return f"Parquet {df.shape[0]}x{df.shape[1]}"
        if ext == ".csv":
            df = pd.read_csv(p, nrows=5, sep=None, engine="python"); return f"CSV, Spalten: {list(df.columns)[:4]}…"
        if ext == ".pdf":
            txt = p.read_bytes(); npg = len(re.findall(rb"/Type\s*/Page[^s]", txt)); return f"PDF, {npg} Seite(n), Raster (kein Text erwartet)"
        if ext == ".xlsx":
            xl = pd.ExcelFile(p); return f"Excel, Blätter: {xl.sheet_names[:3]}"
    except Exception as e:
        return f"Lesefehler: {type(e).__name__}"
    return ""

def bewerte(rel):
    r = rel.lower()
    if "upload/a_daten/acron_stundenwerte_wert" in r: return ("Druck/Temperatur/Durchfluss/Leistung 2025 (stündlich)", "verwendet", "Kalibrierung/Validierung, Heizkurve", "Zeitumstellung: 1 Doppelstunde verworfen; viele Lücken")
    if "upload/a_daten/acron_stundenwerte_minmax" in r: return ("Min/Max zu ACRON-Reihen", "verwendet (später)", "Druckspitzen/-minima", "nicht in Etappe 1 ausgewertet")
    if "acron_signalindex" in r: return ("Zuordnung Signal -> Originaldatei/Station", "verwendet", "Messstellenzuordnung", "-")
    if "import_data_stadtbach_15min" in r: return ("Außentemperatur, V_n-Zähler, WRG (15 min)", "verwendet", "Außentemperatur, Kundenlasten", "Einheiten der V_n-Kanäle ungeklärt")
    if "combined" in r: return ("Aggregierte Kundenlasten (7 gemessen + 17 geschätzt)", "nur Referenz", "Vergleich zur alten Config", "17 Reihen zonal geschätzt -> keine Validierungsziele")
    if "b_plaene" in r: return ("Netzpläne WV640/650/660 (Raster)", "verwendet (Etappe 2)", "Topologie, Armaturen, Höhen, PN", "verkleinerte Raster; Text nicht extrahierbar")
    if "lastg" in r: return ("Lastgänge 2020–2022", "nicht verwendet (Etappe 1)", "evtl. Ausbau-/Lastform", "älter als 2025")
    if r.endswith("stadtbach_topo.yaml") and "configs/paper_2/" in r and "archiev" not in r: return ("Aktuelles YAML-Netz (Baum)", "verwendet als Ausgangspunkt", "Sektions-Skelett, Erzeuger", "Baum statt Masche; Druckniveau nicht messkonform")
    if "archiev" in r and "stadtbach" in r: return ("Ältere Config-Stände", "nicht verwendet", "-", "veraltet; Kommentare widersprüchlich")
    if "storage_geometry" in r: return ("Speichergeometrie (atmosphärische Grube)", "nicht verwendet", "Widerspruch zum Druckspeicher", "atmosph. 95 °C-Grenze; nicht passend")
    if "20250922_stadtbach.ipynb" in r: return ("Alter Pyomo-MILP (keine pandapipes)", "nicht verwendet", "-", "keine pandapipes-Studie vorhanden")
    if "pandapipes" in r: return ("pandapipes-Crosscheck (Memmingen)", "Vorlage", "Vorgehen für Kontrolle", "nur Memmingen")
    if r.endswith(".py") and "stadtbach" in r: return ("Hilfsskript", "nicht verwendet", "-", "-")
    return ("Sonstiges/Dokumentation", "nicht verwendet", "-", "-")

inv["inhalt_geprueft"] = inv.pfad.map(inspect)
b = inv.pfad.map(bewerte)
inv["bedeutung"], inv["status"], inv["begruendung"], inv["qualitaet"] = [b.map(lambda t: t[i]) for i in range(4)]
inv.to_csv(RESULTS/"inventur_dateien.csv", index=False)
with pd.option_context("display.max_colwidth", 70, "display.width", 250):
    display(inv[inv.status.str.startswith(("verwendet","nur","Vorlage"))][["pfad","typ","groesse_kB","inhalt_geprueft","status"]])
print("Status-Übersicht:", inv.status.value_counts().to_dict())

**Befund 4.4 – fehlende Quellen:** Suche nach Stadtbach-DXF/DWG/Shape/GeoJSON: siehe nächste Zelle. Die Pläne liegen nur als verkleinerte Raster-PDFs vor.

In [ ]:
geo = inv[inv.typ.isin(["dxf","dwg","shp","geojson"])]
print("Geometriedateien (Stadtbach):", "keine" if geo.empty else geo.pfad.tolist())
print("DXF im Repo (nur Memmingen erwartet):", [p.relative_to(ROOT).as_posix() for p in ROOT.rglob("*.dxf") if ".git" not in p.parts])
print("Stadtbach-pandapipes-Studie im Repo:", "nein – nur Memmingen-Crosscheck" if not any("pandapipes" in x and "stadtbach" in x.lower() for x in inv.pfad) else "ja")

## 5. Quellen-, Konflikt- und Annahmenkatalog

### 5.1 Annahmenkatalog
Herkunftsklassen: **M** direkt gemessen · **P** Plan/Betriebsunterlage · **Y** YAML/bestehendes Modell · **H** Hersteller · **K** kalibriert · **A** fachlicher Standardwert · **S** Sensitivitätsparameter. Der Katalog wird in späteren Etappen fortgeschrieben.

In [ ]:
katalog = pd.DataFrame([
 # id, parameter, wert, einheit, klasse, quelle, gueltig, unsicherheit, wirkung, sens_bereich
 ("A-01","Druckdefinition ACRON", "Überdruck", "bar", "A","Plausibilität: HKW-RL-Median 8,5 bar; Absolutdruck unwahrscheinlich niedrig für Ruhedruck 3,2–6,5 bar Ü", "2025","hoch: Faktor 1,013 bar","Siedesicherheit, Speicherdruck","abs vs. Ü (Δ=1,013 bar)"),
 ("A-02","Durchflusseinheit ACRON", "m³/h (≈ t/h)", "m³/h", "A","Leistungsbilanz MAN (Abschnitt 6.4)", "2025","mittel","Massenströme","t/h vs. m³/h (Δρ≈3–4 %)"),
 ("A-03","Heizkurve Hauptrechnung", "T_VL=110−T_a, begrenzt 90…120 °C", "°C", "S","Auftraggeber-Vorgabe (vorläufig)", "Winter/Übergang","mittel","Vorlauftemperatur","YAML-Kurve (122→70 °C) als Alternative"),
 ("A-04","Speichertyp", "Druckspeicher, direkt eingebunden", "-", "P","Auftraggeber: bis 120 °C im Winter nötig", "-","-","Betriebsdruck ≥ p_sat(120 °C)+Marge","-"),
 ("A-05","MAN-Anschlusspunkt", "wird in Etappe 2 aus WV650 festgelegt", "-", "A","Auftraggeber: 'Du kannst den MAN-Punkt annehmen'", "-","hoch","Speicherhydraulik","Varianten am Netzknoten"),
 ("A-06","Netz-Spreizung Dimensionierung", "99/60 °C", "°C", "Y","Stadtbach_topo.yaml", "Übergangszeit","mittel","Erzeugerkapazität","Messung 2025 (Abschnitt 8)"),
 ("A-07","Transportgrenzen SL5/SL6/SL7 (separiert)", "750/500/300 t/h", "t/h", "P","WV650 laut Auftrag (noch nicht am Plan geprüft)", "Stand 2022","mittel","Separierter Betrieb","±20 %"),
 ("A-08","Mindest-Ruhedruck bei 120 °C", "MITTE–CITY 6,5 / MITTE 3,2 / SÜD 3,2 / OST 2,5", "bar Ü", "P","WV650 laut Auftrag (noch nicht am Plan geprüft)", "Stand 2022","gering","Druckhaltung","-"),
 ("A-09","Kundendifferenzdruck minimal", "0,7", "bar", "Y","Stadtbach_topo.yaml delta_p_min_consumer_bar", "-","hoch (Messung 0–4 bar)","Druckverletzungen","0,3 … 1,5 bar"),
 ("A-10","Rohrrauheit", "0,5 (YAML) – wirkt dort nicht", "mm", "Y","Stadtbach_topo.yaml; im Framework nur gespeichert, nicht verwendet", "-","hoch","Druckverlust","0,1 … 1,0 mm (Colebrook im Notebook)"),
], columns=["id","parameter","wert","einheit","klasse","quelle_fundstelle","gueltigkeit","unsicherheit","auswirkung","sensitivitaetsbereich"])
katalog.to_csv(RESULTS/"annahmenkatalog.csv", index=False)
with pd.option_context("display.max_colwidth", 60, "display.width", 250): display(katalog[["id","parameter","wert","einheit","klasse","unsicherheit"]])

### 5.2 Konfliktkatalog (Widersprüche zwischen Quellen)
Keine Quelle wird stillschweigend überschrieben; für jeden Widerspruch wird eine Arbeitsannahme festgelegt und die Alternative als Sensitivität gerechnet.

In [ ]:
konflikte = pd.DataFrame([
 ("K-01","Heizkurve","Vorgabe: 120 °C@−10 °C … 90 °C@+20 °C","YAML: 122 °C@−15 °C … 70 °C@+30 °C (+Saisonwerte)","Abschnitt 8: Vergleich mit Messung","Vorgabe für Hauptrechnung, YAML als Sensitivität"),
 ("K-02","Netztopologie","Auftrag/Plan: teilweise vermascht, Sektionen","YAML: 33 Knoten/32 Leitungen, reiner Baum; Header nennt 32/31; Paper-Tabelle nennt 'meshed'","Etappe 2","Sektionsmodell mit Maschen aus WV650/660"),
 ("K-03","Speichertyp","Auftraggeber: Druckspeicher bis 120 °C","storage_geometry.yaml: atmosphärische Grube (95 °C, bis 120 000 m³); Topo-YAML: 10 bar/5 000 m³","Kap. 15","Druckspeicher; atmosphärische Variante entfällt"),
 ("K-04","Speicherenergie","5 000 m³ @ 10 K ≈ 56–58 MWh","YAML-Leiter bis 84,5 MWh (≈ 15 K)","Kap. 15","aus Volumen und ΔT berechnen; 84,5 MWh nicht übernehmen"),
 ("K-05","Druckniveau","Messung 2025: HKW-VL Median 10,7 bar, RL 8,5 bar","YAML: VL-Sollwert 16 bar, RL am Boden 0,5 bar","Kap. 13/14","Messwerte als Anker; YAML-Sollwert verwerfen"),
 ("K-06","Erzeugerleistungen (Kommentare)","YAML-Assets: HKW 127, HWS 32, HWW 104 MW","YAML-Header/Serienmap: HKW 75, HWS 40, HWW 64 MW","Etappe 2/3","Asset-Werte (Üzeit, 99/60 °C); Kommentare veraltet"),
 ("K-07","Hydraulik im Framework","Darcy-Weisbach mit Rauheit (Auftrag)","Framework: konstantes λ=0,02; Δp nur als untere Hüllkurve","Etappe 2","eigener Netzlöser im Notebook"),
], columns=["id","thema","quelle_A","quelle_B","pruefung","arbeitsannahme"])
konflikte.to_csv(RESULTS/"konfliktkatalog.csv", index=False)
with pd.option_context("display.max_colwidth", 55, "display.width", 250): display(konflikte)

## 6. Import und Prüfung der Messdaten
Rohdaten werden **nie überschrieben**: `acron_raw` bleibt unverändert, `acron` ist die bereinigte Arbeitskopie.
Geprüft werden Zeitstempel/Zeitzone/Sommerzeit, Duplikate, Lücken, Nullwerte, eingefrorene Werte, Ausreißer, Vorzeichen und Einheiten.

In [ ]:
# --- 6.1 Import
A = DATA_DIR/"A_Daten"
sig      = pd.read_csv(A/"acron_signalindex.csv")
acron_raw= pd.read_parquet(A/"acron_stundenwerte_wert.parquet")
imp      = pd.read_parquet(A/"Import_Data_stadtbach_15min.parquet").set_index("Datum")

print("ACRON:", acron_raw.shape, acron_raw.index.min(), "→", acron_raw.index.max())
print("Index eindeutig:", acron_raw.index.is_unique, "| Frequenz stündlich:",
      (acron_raw.index.to_series().diff().dropna() == pd.Timedelta("1h")).mean().round(4))
missing_h = pd.date_range(f"{CFG['year']}-01-01", f"{CFG['year']}-12-31 23:00", freq="h").difference(acron_raw.index)
print("Fehlende Stunden im Jahr 2025:", list(missing_h), "(30.03. 02:00 existiert lokal nicht – Sommerzeitbeginn; die doppelte Stunde 26.10. 02:00 wurde beim Upload auf den ersten Wert reduziert; Zeitstempel naiv lokal)")
print("Import 15 min:", imp.shape, imp.index.min(), "→", imp.index.max(), "| eindeutig:", imp.index.is_unique)

def kanal(name):
    n = name
    for k,v in [("Druck_VL","p_VL"),("Druck_RL","p_RL"),("DiffDruck","dp"),("Temp_VL","T_VL"),("Temp_RL","T_RL"),
                ("Durchfluss","m_dot"),("Waermeleistung","Q"),("Drehzahl","n_pumpe")]:
        if k in n: return v
    return "sonstige"
meta = pd.DataFrame({"kanal": [kanal(c) for c in acron_raw.columns]}, index=acron_raw.columns)
meta = meta.join(sig.set_index("key")[["ordner","title"]])
print(meta.kanal.value_counts().to_dict())

In [ ]:
# --- 6.2 Qualitätsanalyse + Bereinigung (regelbasiert, dokumentiert)
def longest_const_run(s):
    s = s.dropna()
    if s.empty: return 0
    return int((s.diff() != 0).cumsum().value_counts().max())

acron = acron_raw.copy()
log = []
p_cols = [c for c in acron.columns if meta.kanal[c] in ("p_VL","p_RL","dp")]
for c in p_cols:
    s = acron[c]
    nb = pd.concat([s.shift(1), s.shift(-1)], axis=1).median(axis=1)         # Nachbarn ±1 h
    if meta.kanal[c] == "dp":
        bad = (s <= 0.0) & (nb > 0.5)
    else:
        bad = (s < 0.5) & (nb > 2.0)                                          # Nullwerte bei intakten Nachbarn
    if bad.any():
        acron.loc[bad, c] = np.nan; log.append((c, "Druck/Δp ≈ 0 bei intakten Nachbarn → NaN", int(bad.sum())))
t_cols = [c for c in acron.columns if meta.kanal[c] in ("T_VL","T_RL")]
for c in t_cols:
    s = acron[c]; bad = (s < 5) | (s > 135)
    if bad.any(): acron.loc[bad, c] = np.nan; log.append((c, "Temperatur außerhalb 5…135 °C → NaN", int(bad.sum())))
m_cols = [c for c in acron.columns if meta.kanal[c] == "m_dot"]
for c in m_cols:
    if c.startswith("HKW_Durchfluss_SL"): continue     # Vorzeichen = Flussrichtung (bewusst erlaubt)
    s = acron[c]; bad = s < -1e-9
    if bad.any(): acron.loc[bad, c] = np.nan; log.append((c, "negativer Durchfluss → NaN", int(bad.sum())))

cleanlog = pd.DataFrame(log, columns=["signal","regel","anzahl"])
print(f"Bereinigungsregeln angewandt auf {len(cleanlog)} Signale, {cleanlog.anzahl.sum()} Werte → NaN")

rows=[]
for c in acron.columns:
    s0, s = acron_raw[c], acron[c]
    rows.append(dict(signal=c, kanal=meta.kanal[c], station=meta.ordner[c],
        abdeckung_pct=round(100*s.notna().mean(),1), luecken_pct=round(100*s.isna().mean(),1),
        min=s.min(), q05=s.quantile(.05), median=s.median(), q95=s.quantile(.95), max=s.max(),
        nullwerte_roh=int((s0==0).sum()), laengster_konst_lauf_h=longest_const_run(s0),
        ausreisser_3sigma=int(((s-s.mean()).abs() > 3*s.std()).sum()) if s.std()>0 else 0,
        bereinigt=int(s0.notna().sum()-s.notna().sum())))
qual = pd.DataFrame(rows).set_index("signal")
qual["eingefroren_verdacht"] = (qual.laengster_konst_lauf_h >= 72) & ~qual.kanal.isin(["n_pumpe"])
# Validierungstauglichkeit: Abdeckung ≥ 80 % und kein Stillstand ≥ 168 h (Pumpendrehzahlen ausgenommen, dort ist Stillstand normal)
qual["validierungstauglich"] = (qual.abdeckung_pct >= 80) & ((qual.laengster_konst_lauf_h < 168) | qual.kanal.isin(["n_pumpe"]))
qual.round(2).to_csv(RESULTS/"datenqualitaet_acron.csv")
cleanlog.to_csv(RESULTS/"bereinigungslog.csv", index=False)

### 6.3 Ergebnis der Qualitätsprüfung

In [ ]:
print("Abdeckung je Kanal (Median über Signale, %):")
print(qual.groupby("kanal").abdeckung_pct.agg(["count","median","min"]).round(1))
print("\nSignale mit < 80 % Abdeckung:")
print(qual[qual.abdeckung_pct < 80][["kanal","station","abdeckung_pct"]].sort_values("abdeckung_pct").to_string())
print("\nEingefrorene Reihen (≥ 72 h konstant, Verdacht auf Sensor-/Übertragungsausfall):")
print(qual[qual.eingefroren_verdacht][["kanal","station","laengster_konst_lauf_h"]].sort_values("laengster_konst_lauf_h", ascending=False).to_string())
print("\nValidierungstauglich (Abdeckung ≥ 80 %, kein Stillstand ≥ 168 h):", int(qual.validierungstauglich.sum()), "von", len(qual), "Signalen; je Kanal:", qual.groupby("kanal").validierungstauglich.sum().to_dict())
print("\nBereinigung (Top 15):"); print(cleanlog.sort_values("anzahl", ascending=False).head(15).to_string(index=False))

### 6.4 Einheiten- und Konsistenzprüfung
* **Durchfluss:** Aus `Q = V̇·cp·ρ·ΔT` wird an Stationen mit Leistung, Durchfluss, VL- und RL-Temperatur die implizite Einheit geprüft (Erwartung: m³/h).
* **Druck:** VL > RL, Δp ≈ VL − RL (soweit alle drei Signale vorhanden).

In [ ]:
# Einheitenprüfung Durchfluss: Q[MW] ≈ V[m³/h]·ρ·cp·ΔT/3.6e6 (ρ≈960, cp≈4,2 kJ/kgK)
rows=[]
for st in sorted(set(meta.ordner)):
    q  = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="Q"]
    v  = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="m_dot"]
    tv = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="T_VL"]
    tr = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="T_RL"]
    if q and v and tv and tr:
        d = pd.concat([acron[q[0]], acron[v[0]], acron[tv[0]]-acron[tr[0]]], axis=1, keys=["Q","V","dT"]).dropna()
        d = d[(d.Q>0.05)&(d.V>5)&(d.dT>5)]
        if len(d) > 100:
            q_calc = d.V*960*4.2*d.dT/3.6e6
            rows.append((st, len(d), round((d.Q/q_calc).median(),3), round((d.Q/q_calc).quantile(.1),2), round((d.Q/q_calc).quantile(.9),2)))
einh = pd.DataFrame(rows, columns=["station","n","Q_gemessen/Q_aus_V·ΔT (Median)","q10","q90"])
print(einh.to_string(index=False))
print("\n→ Kundenstationen: Verhältnis ≈ 1,04 (Näherungskonstanten ρ=960, cp=4,2) bestätigt m³/h. Abweichend: HKW (≈15×), HWW sekundär (≈2,2×), GT-Ost (≈0,7×) – dort ist Definition von Durchfluss bzw. Leistung (Teilstrom/Bezugsstelle) ungeklärt → Rückfrage; diese Signale werden nicht als Validierungsziel verwendet.")

# Druckkonsistenz
pc = []
for st in sorted(set(meta.ordner)):
    pv = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="p_VL"]
    pr = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="p_RL"]
    dd = [c for c in acron.columns if meta.ordner[c]==st and meta.kanal[c]=="dp"]
    if pv and pr:
        d = (acron[pv[0]]-acron[pr[0]]).dropna()
        row = dict(station=st, VL_kleiner_RL_pct=round(100*(d<0).mean(),1), dp_VL_RL_median=round(d.median(),2))
        if dd:
            e = (acron[dd[0]]-(acron[pv[0]]-acron[pr[0]])).dropna(); row["dp_gemessen_minus_(VL−RL)_median"]=round(e.median(),2)
        pc.append(row)
print(); print(pd.DataFrame(pc).to_string(index=False))

## 7. Explorative Datenanalyse
Außentemperatur, Netztemperaturen, Drücke am HKW und Differenzdrücke an den Kundenstationen 2025.

In [ ]:
Ta = imp["outdoor_temp_C"].resample("1h").mean()
Ta = Ta.reindex(acron.index)
fig, ax = plt.subplots(4,1, figsize=(12,10), sharex=True)
ax[0].plot(Ta, color=COL["grau"], lw=.6); ax[0].set_ylabel("Außentemp. [°C]")
ax[1].plot(acron["HKW_Temp_VL"], color=COL["vl"], lw=.6, label="HKW Vorlauf"); ax[1].plot(acron["HKW_Temp_RL"], color=COL["rl"], lw=.6, label="HKW Rücklauf")
ax[1].set_ylabel("Temperatur [°C]"); ax[1].legend(loc="upper right", ncol=2)
ax[2].plot(acron["HKW_Druck_VL"], color=COL["vl"], lw=.6, label="HKW Druck VL"); ax[2].plot(acron["HKW_Druck_RL"], color=COL["rl"], lw=.6, label="HKW Druck RL")
ax[2].set_ylabel("Druck [bar Ü]"); ax[2].legend(loc="upper right", ncol=2)
ax[3].plot(acron["HKW_Durchfluss_gesamt"], color=COL["mess"], lw=.6); ax[3].set_ylabel("HKW Durchfluss ges. [m³/h]")
fig.suptitle("Netzzustand am HKW, 2025", y=.995); fig.tight_layout(); fig.savefig(FIG_DIR/"07_hkw_zeitreihen.png"); plt.show()

In [ ]:
# Differenzdruck-Dauerlinien der Kundenstationen: kritische Stellen
dp_cols = [c for c in acron.columns if meta.kanal[c]=="dp" and meta.ordner[c] not in ("PSS","PSW","HKW","BMHKW","GT-Ost","AVA","Heizwerk_Sued","HWW_prim_Einsp")]
dpq = pd.DataFrame({c:{"q01":acron[c].quantile(.01),"median":acron[c].median(),"anteil_<0,7bar_%":100*(acron[c]<0.7).mean()} for c in dp_cols}).T.round(2)
dpq = dpq.sort_values("q01")
display(dpq.head(12))
fig, ax = plt.subplots(figsize=(9,5))
for c in dpq.index[:6]:
    s = acron[c].dropna().sort_values(ascending=False).reset_index(drop=True)
    ax.plot(np.linspace(0,100,len(s)), s, label=c.replace("_DiffDruck",""))
ax.axhline(0.7, color=COL["warn"], ls="--", label="0,7 bar (YAML-Annahme)")
ax.set_xlabel("Zeitanteil [%]"); ax.set_ylabel("Differenzdruck [bar]"); ax.set_title("Differenzdruck-Dauerlinien, sechs kritischste Stationen"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR/"07_dp_dauerlinien.png"); plt.show()

## 8. Prüfung der Heizkurve
Vergleich von (a) der **Vorgabe** (Auftrag), (b) der **YAML-Kurve**, (c) der **gemessenen** Vorlauftemperatur am Erzeugeraustritt und (d) der **statistisch aus Messdaten** ermittelten Kurve.

Vorgabe: $T_{VL,soll}=\min\big(120\,^\circ\mathrm{C},\ \max(90\,^\circ\mathrm{C},\ 110\,^\circ\mathrm{C}-T_{a})\big)$
YAML: linear von 122 °C bei −15 °C bis 70 °C bei +30 °C.

In [ ]:
def hk_vorgabe(Ta):
    return np.clip(110.0 - np.asarray(Ta, float), 90.0, 120.0)
def hk_yaml(Ta):
    return np.clip(122.0 - (122.0-70.0)/(30.0-(-15.0))*(np.asarray(Ta, float)+15.0), 70.0, 122.0)

# Tests: Endpunkte
assert hk_vorgabe(-10)==120 and hk_vorgabe(20)==90 and hk_vorgabe(-20)==120 and hk_vorgabe(30)==90
assert abs(hk_yaml(-15)-122)<1e-9 and abs(hk_yaml(30)-70)<1e-9
print("Heizkurven-Endpunkttests bestanden.")

hk = pd.DataFrame({"Ta":Ta, "VL_HKW":acron["HKW_Temp_VL"], "VL_MAN":acron.get("MAN_Temp_VL_Station"),
                   "VL_PPS":acron.get("PPS_Temp_VL"), "VL_PSW":acron.get("PSW_Temp_VL"),
                   "V_ges":acron["HKW_Durchfluss_gesamt"]}).dropna(subset=["Ta","VL_HKW"])
hk["soll_vorgabe"] = hk_vorgabe(hk.Ta); hk["soll_yaml"] = hk_yaml(hk.Ta)
hk["mon"] = hk.index.month; hk["stunde"] = hk.index.hour
hk["saison"] = hk.mon.map(lambda m: "Winter" if m in (12,1,2) else "Frühjahr" if m in (3,4,5) else "Sommer" if m in (6,7,8) else "Herbst")
hk["last"] = pd.qcut(hk.V_ges, 3, labels=["niedrig","mittel","hoch"])

# Statistische Kurve: Median/Quantile je 2-K-Klasse + lineare Anpassung (−10…+20 °C)
hk["bin"] = (np.floor(hk.Ta/2)*2+1)
st = hk.groupby("bin").VL_HKW.agg(n="count", q10=lambda s: s.quantile(.1), median="median", q90=lambda s: s.quantile(.9)).query("n>=20")
sel = st.loc[(st.index>=-10)&(st.index<=20)]
k, d0 = np.polyfit(sel.index, sel["median"], 1)
print(f"Statistische Kurve (Median, −10…20 °C): T_VL = {d0:.1f} {k:+.3f}·T_a  → bei −10 °C: {d0+k*-10:.1f} °C, bei +20 °C: {d0+k*20:.1f} °C")
for nm, f in (("Vorgabe",hk_vorgabe),("YAML",hk_yaml)):
    r = hk.VL_HKW - f(hk.Ta); print(f"{nm:8s}: Bias {r.mean():+.1f} K, RMSE {np.sqrt((r**2).mean()):.1f} K, MAE {r.abs().mean():.1f} K")

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,5))
xs = np.linspace(-13,32,100)
ax[0].scatter(hk.Ta, hk.VL_HKW, s=3, alpha=.15, color=COL["grau"], label="Messung HKW, stündlich")
ax[0].fill_between(st.index, st.q10, st.q90, color=COL["vl"], alpha=.15, label="10–90 %-Band")
ax[0].plot(st.index, st["median"], "o-", color=COL["vl"], ms=3, label="Median je 2-K-Klasse")
ax[0].plot(xs, hk_vorgabe(xs), color=COL["ok"], lw=2, label="Vorgabe (Auftrag)")
ax[0].plot(xs, hk_yaml(xs), color=COL["warn"], lw=2, ls="--", label="YAML-Kurve")
ax[0].set(xlabel="Außentemperatur [°C]", ylabel="Vorlauftemperatur [°C]", ylim=(60,130), title="Heizkurve: Vorgabe, YAML und Messung")
ax[0].legend(fontsize=8)
res = (hk.VL_HKW - hk.soll_vorgabe)
hk.assign(res=res).boxplot(column="res", by="saison", ax=ax[1], grid=False)
ax[1].axhline(0, color="k", lw=.8); ax[1].set(title="Abweichung Messung − Vorgabe nach Jahreszeit", xlabel="", ylabel="ΔT_VL [K]"); plt.suptitle("")
fig.tight_layout(); fig.savefig(FIG_DIR/"08_heizkurve.png"); plt.show()

In [ ]:
# Abweichungen nach Tageszeit, Lastniveau, Netzbereich
hk["res"] = hk.VL_HKW - hk.soll_vorgabe
print("Nach Tageszeit (Median Abweichung, K):")
print(hk.groupby(pd.cut(hk.stunde,[-1,5,11,17,23],labels=["Nacht 0–5","Vormittag 6–11","Nachmittag 12–17","Abend 18–23"]),observed=True).res.median().round(1).to_string())
print("\nNach Lastniveau (HKW-Durchfluss-Terzile):"); print(hk.groupby("last",observed=True).res.median().round(1).to_string())
bereich = pd.DataFrame({b: (hk[c] - hk.soll_vorgabe) for b,c in (("HKW (Mitte)","VL_HKW"),("MAN","VL_MAN"),("Pumpstation Süd","VL_PPS"),("Pumpstation West","VL_PSW")) if hk[c].notna().any()})
print("\nNach Netzbereich (Median Abweichung zur Vorgabe, K; Winter):")
print(bereich[hk.saison=="Winter"].median().round(1).to_string())
kalt = hk[hk.Ta < -5]
print(f"\nKältester Bereich (T_a < −5 °C, n={len(kalt)} h): gemessener Vorlauf Median {kalt.VL_HKW.median():.1f} °C, max {kalt.VL_HKW.max():.1f} °C; Vorgabe 115–120 °C")
print("Spitzen ≥ 120 °C:", int((hk.VL_HKW>=120).sum()), "Stunden; ≥ 122 °C:", int((hk.VL_HKW>=122).sum()), "Stunden")

### 8.4 Zwischenbewertung Heizkurve (aus den Rechenergebnissen oben)

In [ ]:
rm = {n: float(np.sqrt(((hk.VL_HKW - f(hk.Ta))**2).mean())) for n, f in (("Vorgabe", hk_vorgabe), ("YAML", hk_yaml))}
bias = {n: float((hk.VL_HKW - f(hk.Ta)).mean()) for n, f in (("Vorgabe", hk_vorgabe), ("YAML", hk_yaml))}
best = min(rm, key=rm.get)
print(f"Hauptrechnung: Heizkurve '{best}' (RMSE {rm[best]:.1f} K, Bias {bias[best]:+.1f} K) gegenüber '{[k for k in rm if k!=best][0]}' (RMSE {max(rm.values()):.1f} K, Bias {bias[[k for k in rm if k!=best][0]]:+.1f} K).")
print(f"Statistische Kurve T_VL = {d0:.1f} {k:+.3f}·T_a weicht bei −10 °C um {d0+k*-10-120:+.1f} K und bei +20 °C um {d0+k*20-90:+.1f} K von der Vorgabe ab.")
print("Die YAML-Kurve wird als Sensitivität mitgeführt (Konflikt K-01). Es gab 2025 keinen Speicherbetrieb; eine Abhängigkeit vom Speicher entfällt.")
print("Hinweis: Nur 182 Stunden mit T_a < −5 °C und minimal −12,9 °C im Jahr 2025 – der Auslegungsfall (−14 … −16 °C) ist NICHT durch Messung abgedeckt und wird durch die Kurve extrapoliert.")

---
**Ende Etappe 1.** Folgt: Kapitel 9–10 (Netzpläne, kanonisches Netzmodell) – danach Halt zur Prüfung der abgelesenen Topologie.

## 9. Analyse der Netzpläne und der YAML-Topologie
**Quellen:** Arbeitsblätter *WV 640 Bl. 3* (Erzeugerausspeisungen, Stand 14.06.2023), *WV 650* (Netz Mitte/Süd/Nordost/Ost, Stand 25.03.2022 bzw. Datei 2022-10-28) und *WV 660* (Netz West/Nordwest, Stand 25.03.2022).
Die Pläne liegen nur als **verkleinerte Rasterbilder** vor. Alle Werte in diesem Kapitel wurden **manuell bzw. per OCR aus den Bildern abgelesen** und sind hier als Datentabellen hinterlegt (Herkunftsklasse **P**). Ablesefehler sind möglich – die Tabellen sind zur Prüfung durch den Auftraggeber gedacht.

### 9.1 Erzeuger, Pumpstationen und Druckhaltung (WV 640 Bl. 3)

In [ ]:
# Höhen [m NN] = Bezugshöhe der Anlage; Ruhedruck [bar Ü] und Abschaltdrücke [bar Ü] gemäß WV640 Bl.3
erzeuger = pd.DataFrame([
 # anlage, z_NN, dp_intern, P_AT14, m_AT14, T_VL/T_RL AT14, P_UeZ, m_UeZ, T_VL/T_RL UeZ, Tmax_VL, Pumpen, Anordnung, Ruhedruck, Ruhedruck_bez_NN, ab_RL_lt, ab_VL_gt, Bezug_NN_Abschalt
 ("HKW",      473.77, 1.1, 127.0, 2350, "107/60 (Parallelbetr.)", None, None, None, 128, "2x1000 + 2x900 t/h @ 73,5 m FIS", "im VL", 8.4, 474.68, None, None, None),
 ("AVA",      466.69, 0.8,  40.0,  530, "125/60",  45.0,  990, "99/60", 128, "2x500 t/h @ 90 m FIS",  "im RL", 7.5, 467.84, 5.0, 19.5, 467.84),
 ("GT-Ost",   467.98, 1.3,  31.0,  410, "125/60",  35.0,  720, "102/60",128, "2x725 t/h @ 90 m FIS",  "im RL", 6.8, 474.98, 4.4, 20.5, 474.98),
 ("BM-HKW",   467.08, 0.9,  13.0,  190, "120/60",  14.5,  400, "91/60", 120, "2x206 t/h @ 115 m FIS", "im RL", None, None, 4.8, 22.0, 473.75),
 ("HWS",      490.51, 0.8,  40.0,  550, "128/65",  32.0,  550, "110/60",128, "3x280 m3/h @ 58,5 m FIS","im RL", 6.8, 488.16, 4.0, 12.0, 488.05),
 ("HWW",      477.99, None, 64.0,  960, "120/65 (2 Kessel + 2 WÜ Prim-Einsp.)", 24.0, 516, "100/60 (2 WÜ Prim-Einsp.)", 120, "3x480 m3/h @ 75 m FIS", "im RL", 6.8, 475.18, 4.6, None, 475.18),
], columns=["anlage","z_NN_m","dp_intern_bar","P_max_AT-14_MW","m_max_AT-14_t/h","T_VL/T_RL_AT-14","P_max_UeZeit_MW","m_max_UeZeit_t/h","T_VL/T_RL_UeZeit",
            "Tmax_VL_C","Umwaelzung_installiert","Pumpenanordnung","Ruhedruck_bar_Ue","Ruhedruck_Bezug_mNN","abschalt_RL_unter_bar_Ue","abschalt_VL_ueber_bar_Ue","abschalt_Bezug_mNN"])
pumpst = pd.DataFrame([
 ("PSW", 471.55, "P1+2: 700 m3/h @ 38 m FIS (kein Parallelbetr.); P3: 325 m3/h @ 30 m FIS", "im VL, Richtung HWW", 0.2, 5.0, 18.0, 471.16),
 ("PSS", 481.45, "P1+2: 2x429 m3/h @ 40 m FIS (Richtung HWS); P3: 215 m3/h @ 27 m FIS (Richtung HKW)", "im VL", 0.2, 5.0, 18.0, 482.00),
], columns=["pumpstation","z_NN_m","Umwaelzung_installiert","Anordnung","dp_intern_bar","abschalt_Saugseite_unter_bar_Ue","abschalt_Druckseite_ueber_bar_Ue","Bezug_mNN"])
extras = pd.DataFrame([
 ("HKW","Max. Differenzdruck Austritt Werk","4,0 bar"), ("HKW","Ruhedruckbalken min/normal/max","6,1 / 8,4 / 9,1 bar Ü (Bezug 474,68 m NN)"),
 ("HKW","max. P (Umlenkbetrieb, Parallelb. mit Ost)","143 MW"), ("Ost-Erzeuger","max. Netz-Massenstrom parallel AVA/GT/BM","1550 t/h (SL5 750, SL6 500, SL7 300)"),
 ("Ost-Erzeuger","max. Differenzdruck Austritt Werk","7,5 bar"), ("HWW","Prim-Netz max. Vorlauftemperatur","110 °C"),
 ("HWW","WÜ Prim-Ausspeisung AT-14","31 MW/485 t/h + 9 MW/140 t/h (120/65 °C)"), ("HWW","WÜ Prim-Ausspeisung ÜZeit","12 MW/258 t/h (100/60 °C)"),
 ("HWW","2 Kessel (sekundär, direkt)","40 MW/625 t/h (120/65 °C)"), ("HWW","Differenzdruckbereich Eintritt WÜ","0,7 … 2,5 bar (max. 3,0); Überströmventil öffnet bei Δp > 3,5 bar"),
], columns=["anlage","groesse","wert"])
erzeuger.to_csv(RESULTS/"plan_erzeuger.csv", index=False); pumpst.to_csv(RESULTS/"plan_pumpstationen.csv", index=False)
with pd.option_context("display.width",250,"display.max_columns",30,"display.max_colwidth",40):
    display(erzeuger[["anlage","z_NN_m","P_max_AT-14_MW","P_max_UeZeit_MW","Tmax_VL_C","Pumpenanordnung","Ruhedruck_bar_Ue","abschalt_RL_unter_bar_Ue","abschalt_VL_ueber_bar_Ue"]])
    display(pumpst); display(extras)

**Konsistenzprüfungen gegen die YAML-Config und die Messung.**
* Die YAML-Assets (127/35/14,5/45/32/104 MW) entsprechen den Übergangszeit-Werten des Plans (HKW 127; GT-Ost 35; BM-HKW 14,5; AVA 45; HWS 32; HWW 24+… ).
* **HWW 104 MW (YAML)** = 64 MW (2 Kessel + 2 WÜ Prim-Einspeisung) + 40 MW (2 Kessel sekundär). Nur die 64 MW bzw. 24 MW (ÜZeit) speisen in das *Primärnetz* ein.
* Die **Ruhedruckdaten** (HKW 8,4 bar Ü) bestätigen die Überdruck-Annahme A-01 durch den Vergleich mit der Messung im nächsten Abschnitt.

In [ ]:
# --- Ruhedruck-Druckhöhe: H = z_Bezug + p/(ρ g)  (Netzpegel, Bezug für Höhenvergleiche)
RHO_COLD, G = 983.0, 9.81           # Wasser ~60 °C
def p_to_head_m(p_bar): return p_bar*1e5/(RHO_COLD*G)
rd = erzeuger.dropna(subset=["Ruhedruck_bar_Ue"]).copy()
rd["Druckhoehe_mNN"] = rd.Ruhedruck_Bezug_mNN + p_to_head_m(rd.Ruhedruck_bar_Ue)
print("Ruhedruck-Druckhöhe (Netzpegel) je Anlage [m NN]:"); print(rd[["anlage","Ruhedruck_bar_Ue","Ruhedruck_Bezug_mNN","Druckhoehe_mNN"]].round(1).to_string(index=False))
print("\nHinweis: Inselbetrieb-/Notbetrieb-Ruhedrücke einzelner Anlagen sind Werte für den *getrennten* Betrieb; im Verbund gilt der Ruhedruckbalken am HKW.")

# --- Messung 2025 gegen Plan: Ruhedruck HKW und Abschaltgrenzen der Erzeuger
def stat(col, f): return float(f(acron[col].dropna())) if col in acron else np.nan
chk = []
chk.append(("HKW Rücklauf (Ruhedruckbalken min/max)", "6,1 … 9,1", stat("HKW_Druck_RL",np.min), stat("HKW_Druck_RL",np.max), stat("HKW_Druck_RL",np.median), "Plan-Fenster liegt um die Messung -> Überdruck bestätigt"))
for sig, lim, an in (("AVA_Druck_RL",5.0,"AVA"),("GT-Ost_Druck_RL",4.4,"GT-Ost"),("BMHKW_Druck_RL",4.8,"BM-HKW"),("Heizwerk_Sued_Druck_RL",4.0,"HWS"),("HWW_Druck_RL",4.6,"HWW (sekundär, Messort ungeklärt)")):
    chk.append((f"{an}: Rücklauf-Untergrenze {lim} bar", f"≥ {lim}", stat(sig,np.min), stat(sig,np.max), stat(sig,np.median), f"Reserve nach unten: {stat(sig,np.min)-lim:+.1f} bar"))
for sig, lim, an in (("Heizwerk_Sued_Druck_RL",12.0,"HWS"),("HWW_Druck_RL",8.3,"HWW (sekundär, Messort ungeklärt)")):
    chk.append((f"{an}: Rücklauf-Obergrenze {lim} bar", f"≤ {lim}", stat(sig,np.min), stat(sig,np.max), stat(sig,np.median), f"Reserve nach oben: {lim-stat(sig,np.max):+.1f} bar"))
for sig, lim, an in (("AVA_Druck_VL",19.5,"AVA"),("GT-Ost_Druck_VL",20.5,"GT-Ost"),("BMHKW_Druck_VL",22.0,"BM-HKW"),("PSW_Druck_VL_nach_Pumpe",18.0,"PSW Druckseite"),("PPS_Druck_VL_nach_Pumpe",18.0,"PSS Druckseite")):
    chk.append((f"{an}: Vorlauf-/Druckseiten-Obergrenze {lim} bar", f"≤ {lim}", stat(sig,np.min), stat(sig,np.max), stat(sig,np.median), f"Reserve nach oben: {lim-stat(sig,np.max):+.1f} bar"))
chk = pd.DataFrame(chk, columns=["Größe","Plan-Grenze","Messung_min","Messung_max","Messung_median","Bewertung"])
display(chk.round(2)); chk.round(2).to_csv(RESULTS/"pruefung_druck_plan_vs_messung.csv", index=False)

### 9.2 Sektionen und Netzbereiche (WV 650 / WV 660)
Die vier HKW-Stammleitungen **SL1–SL4** versorgen im *separierten* Betrieb je eine Sektion. Die Ost-Erzeuger (AVA, GT-Ost, BM-HKW) speisen über **SL5, SL6, SL7** in die Sektionen SL1 bzw. SL3 ein (gesicherte Transportkapazität 750/500/300 t/h). Das Westnetz (PN16) wird über drei Leitungen SEK1–SEK3 aus dem Heizwerk West versorgt.

In [ ]:
# Netzvolumen [m3], Kundenanlagen [Stck], Anschlussleistung [MW] laut Plan (Stand 17.03.2022)
sek650 = pd.DataFrame([
 ("SL1","SL1-West",239,9,2.7),("SL1","SL1-Mitte",315,10,1.9),("SL1","SL1-Nord",0,0,0.0),("SL1","SL1-Ost",425,14,1.6),
 ("SL1","SL5-West mit MAN",123,1,32.0),("SL1","SL5-Mitte",480,4,2.4),("SL1","SL5-Ost",613,23,8.2),
 ("SL2","DOM-West",231,77,10.4),("SL2","DOM-Mitte",222,168,28.1),("SL2","DOM-Ost",205,89,13.4),
 ("SL2","CITY-West",179,84,29.4),("SL2","CITY-Mitte",214,166,33.5),("SL2","CITY-Ost",59,35,6.7),("SL2","CITY-Nord",39,71,14.5),("SL2","CITY-Süd",286,67,14.7),
 ("SL3","STAWA-West",160,67,13.5),("SL3","STAWA-Nord",164,25,8.0),("SL3","STAWA-Ost",326,50,16.7),("SL3","STAWA-Süd",141,56,12.9),("SL3","STAWA-Don Bosco",178,17,7.7),
 ("SL3","SL6-Ost",751,25,5.8),("SL3","SL6-Süd",529,73,3.6),("SL3","SL7-West",99,0,0.0),("SL3","SL7-Mitte",281,7,10.1),("SL3","SL7-Ost",372,0,2.3),
 ("SL4","SL4-Nord",346,11,2.1),("SL4","SL4-Süd",259,26,10.0),("SL4","ULRICH-Ost",63,78,9.1),("SL4","ULRICH-West",127,90,17.4),
 ("SL4","SÜD-Hochfeld-West",217,35,4.2),("SL4","SÜD-Hochfeld-Ost",388,27,9.3),("SL4","SÜD-Technopark",115,21,8.0),("SL4","SÜD-UNI",98,2,8.3),
 ("SL4","SÜD-Univiertel-Süd",141,38,19.1),("SL4","SÜD-Univiertel-Mitte",19,7,4.3),("SL4","SÜD-Univiertel-Nord",124,22,10.9),("SL4","SÜD-RWS",237,6,3.7),
 ("SL4","SÜD-Inno-Park",61,6,4.1),("SL4","SÜD-Antonsviertel",49,4,1.4)],
 columns=["sektion","teilgebiet","netzvol_m3","anlagen","p_anschluss_MW"])
sek650["netz"] = "Verbund (PN25)"
sek660 = pd.DataFrame([
 ("SEK1","Neusäß",30,14,1.6),("SEK1","UNI-MED",21,1,2.5),("SEK1","SEK1-West",356,12,23.0),("SEK1","SEK1-Mitte",147,9,2.8),("SEK1","SEK1-Süd",236,100,6.8),("SEK1","SEK6-Gesamt",131,31,8.1),
 ("SEK2","SEK2-Ost",127,128,4.0),("SEK2","SEK2-Nord",132,44,7.2),("SEK2","SEK5-Nord",54,15,0.5),("SEK2","SEK5-Süd",41,29,3.3),
 ("SEK3","SEK3-Nord",228,58,5.8),("SEK3","SEK3-Süd",151,181,8.0),("SEK3","SEK4-Gesamt",127,37,2.1)],
 columns=["sektion","teilgebiet","netzvol_m3","anlagen","p_anschluss_MW"])
sek660["netz"] = "West/Nordwest (PN16)"
sek = pd.concat([sek650, sek660], ignore_index=True)

# Summenprüfung gegen die im Plan ausgewiesenen Sektions-/Gesamtsummen
plan_summen = {"SL1":(2195,61,48.9),"SL2":(1434,757,150.7),"SL3":(3000,313,80.6),"SL4":(2244,373,111.9),"SEK1":(921,167,44.8),"SEK2":(354,216,15.0),"SEK3":(506,276,15.9)}
pruef = sek.groupby("sektion")[["netzvol_m3","anlagen","p_anschluss_MW"]].sum().round(1)
pruef = pruef.join(pd.DataFrame(plan_summen, index=["vol_plan","anl_plan","MW_plan"]).T)
pruef["Δ_MW"] = (pruef.p_anschluss_MW - pruef.MW_plan).round(1)
display(pruef)
tot650 = sek650.p_anschluss_MW.sum(); tot660 = sek660.p_anschluss_MW.sum()
print(f"Summe Verbund: {tot650:.1f} MW (Plan 392,0) | Netzvolumen {sek650.netzvol_m3.sum()} m3 (Plan 8.874) | Summe West: {tot660:.1f} MW (Plan 75,8)")
assert abs(tot650-392.0) < 0.6 and abs(tot660-75.8) < 0.3, "Transkriptionsfehler in Sektionstabelle"
sek.to_csv(RESULTS/"plan_sektionen.csv", index=False)
print("→ Leistungen und Netzvolumina konsistent mit den Plan-Summen (Rundung ±0,1 MW).")
print("Hinweis: Die Anlagenzahl der Sektion SL3 ist im Plan intern uneinheitlich (Teilgebiete 215+98+13 = 326, Sektionssumme 313); die Leistung ist davon nicht betroffen.")

**Auffällig:** *SL5-West mit MAN* ist im Plan **eine einzige Kundenanlage mit 32,0 MW Anschlussleistung** – das ist 8,2 % der Gesamtanschlussleistung (392 MW) des Verbundnetzes und sitzt am Ende der Trasse SL5, unmittelbar vor der Einbindung in die SL1-Sektion (Armaturen 342.1–342.3 „Stadtbachstraße, MAN östlich der Pforte“).

### 9.3 Armaturen und Trennstellen
Transkription der Tabelle „Einbauorte der Armaturen zur Netztrennung“ (WV650: 103 Zeilen, WV660: 21 Zeilen; per OCR mit Plausibilisierung, **kosmetische OCR-Reste in den Straßennamen möglich**, Nummern, DN und Normalstatus wurden geprüft).

In [ ]:
ARMATUREN_ROWS = [["01", "1529b", "329EZ", "Franziskanergasse9 (HKW SL1)", "MAS", 350, "offen", "WV650"], ["02", "1529b", "329EZ", "Franziskanergasse9 (HKW SL2)", "MAH", 350, "offen", "WV650"], ["03", "1529b", "329EZ", "Franziskanergasse9 (HKWSL3)", "MAH", 350, "offen", "WV650"], ["04", "1529b", "329EZ", "Franziskanergasse9 (HKWSL4)", "MAH", 400, "offen", "WV650"], ["001.2", "1428b", "001S", "Thomm Straße (Ecke Herwart Straße)", "MAH", 400, "offen", "WV650"], ["007.1", "1527d", "007S", "Senkelbach Straße", "MAH", 400, "offen", "WV650"], ["035.1", "1528c", "035S", "Jesuitengasse13", "MAH", 200, "offen", "WV650"], ["042.1", "1528d", "042S", "Hoher Weg30", "MAH", 350, "offen", "WV650"], ["050.1", "1529d", "050S", "Bei Sankt Max", "MAH", 300, "offen", "WV650"], ["050.2", "1529d", "050S", "Bei Sankt Max", "MAH", 250, "offen", "WV650"], ["080.2", "1626d", "080S", "Schletterer Straße (Richtung SL2)", "MAH", 350, "offen", "WV650"], ["080.3", "1626d", "080S", "Schletterer Straße (Richtung Fuggerschule)", "MAH", 150, "offen", "WV650"], ["105.2", "1627c", "105S", "Frölich Straße (Richtung Diako)", "MAH", 125, "offen", "WV650"], ["105.3", "1627c", "105S", "Frölich Straße (Richtung Hauptbahnhof)", "MAH", 300, "offen", "WV650"], ["112.1", "1627d", "112G", "Gutenberg Straße 1 (Maria-Theresia-Gymn.im Keller)", "MAH", 150, "offen", "WV650"], ["114.2", "1628a", "114S", "Kaserm Straße (Richtung Theater Straße)", "MAH", 300, "offen", "WV650"], ["114.3", "1628a", "114S", "Kasern Straße (Richtung Schletter Straße)", "MAH", 350, "offen", "WV650"], ["141.1", "1628b", "141G", "Hoher Weg 1 (swa-Betriebsgebaude im Keller)", "MAH", 80, "zu", "WV650"], ["165.1", "1628d", "165G", "Anna Straße3", "MAH", 80, "offen", "WV650"], ["166.1", "1729d", "166S", "Forster Straße", "MAH", 250, "offen", "WV650"], ["170.1", "1628d", "170G", "Rathausplatz8", "MAH", 65, "offen", "WV650"], ["185.1", "1629a", "185G", "Leonhardsberg (in der Tiefgarage)", "MAH", 200, "offen", "WV650"], ["187.1", "1629a", "187G", "Leonhardsberg 15 (Stadtbad Heizzentrale)", "MAH", 250, "zu", "WV650"], ["195.1", "1628a", "195G", "Grottenau6", "MAH", 80, "offen", "WV650"], ["203.1", "1629c", "203G", "Barfufer Straße (Brücke-Nord)", "MAH", 200, "offen", "WV650"], ["204.1", "1629c", "204G", "Rathausplatz2a (Verwaltungsgebäude Il)", "MAH", 200, "offen", "WV650"], ["207.1", "1629c", "207S", "Barfußer Straße (Brücke-Suid)", "AH", 200, "zu", "WV650"], ["210.1", "1629d", "210S", "Au Berer Gang", "MAH", 150, "zu", "WV650"], ["214.1", "1630a", "214G", "Rauwolff Straße5", "MAH", 100, "offen", "WV650"], ["230.1", "1728c", "230G", "Halder Straße 3 (Stadtsparkasse)", "MAH", 125, "offen", "WV650"], ["233.1", "1728a", "233S", "Fugger Straße2 (Mc Donalds)", "MAH", 150, "offen", "WV650"], ["252.1", "1728c", "252S", "Konrad-Adenauer-Allee 13-15", "MAH", 150, "offen", "WV650"], ["253.1", "1728d", "253S", "Hall Straße (Ecke K.-Adenauer-Allee 21 im Gehweg)", "MAH", 200, "offen", "WV650"], ["275.1", "1728d", "275G", "Maximilian Straße 69 (Standesamt)", "MAH", 200, "offen", "WV650"], ["276.1", "1729d", "276S", "Am Schwall (Schacht beim Glascontainer)", "MAH", 200, "offen", "WV650"], ["278.1", "1729d", "278S", "Am Schwal (Brücke)", "MAK", 350, "offen", "WV650"], ["291.1", "1829d", "291G", "Beim Rabenbad6", "MAH", 150, "offen", "WV650"], ["294.1", "1929b", "294S", "Rote-Torwall-Straße-Trennung SL4", "AS", 350, "offen", "WV650"], ["297.1", "2029b", "297P", "An der Hochschule (Pumpstation Süd)", "MAH", 350, "offen", "WV650"], ["300.1", "1628a", "300G", "Hafnerberg2", "MAH", 150, "offen", "WV650"], ["325.1", "1729a", "325G", "Hunoldsgraben15 (inder Tiefgarage)", "MAH", 125, "offen", "WV650"], ["338.1", "1531c", "338G", "Lechhauser Straße 20", "MAH", 250, "offen", "WV650"], ["338.2", "1531c", "338G", "Lechhauser Straße20", "MAH", 350, "offen", "WV650"], ["338.3", "1531c", "338G", "Lechhauser Straße20", "MAH", 250, "offen", "WV650"], ["342.1", "1329c", "342S", "Stadtbach Straße (MAN östlich der Pforte)", "MAH", 400, "offen", "WV650"], ["342.2", "1329c", "342S", "Stadtbach Straße (MAN östlich der Pforte)", "MAH", 300, "offen", "WV650"], ["342.3", "1329c", "342S", "Stadtbach Straße (MAN östlich der Pforte)", "MAH", 400, "offen", "WV650"], ["346.1", "1625d", "346P", "Holzbach Straße14 (Pumpstation West)", "MAH", 350, "offen", "WV650"], ["368.1", "1731c", "368E", "Nagahama-Alle (Am Provinopark 1a)", "AH", 250, "offen", "WV650"], ["441.5", "1628c", "441S", "Fugger Straße (ehem.Fu Bgangertunnel)", "MAH", 150, "offen", "WV650"], ["476.1", "2428d", "476G", "Universitats Straße 10 (UNI)", "MAH", 300, "offen", "WV650"], ["502.1", "2029d", "502S", "Scherlin Straße (Haltestelle)", "MAH", 350, "offen", "WV650"], ["502.2", "2029d", "502S", "Schertlin Straße (Haltestelle)", "MAH", 250, "offen", "WV650"], ["515.1", "0934b", "515EZ", "Beim Grenzgraben 10 (GT-Ost)", "MAS", 350, "offen", "WV650"], ["517.1", "2329a", "517S", "Alter Postweg 86a (RWS)", "MAH", 350, "offen", "WV650"], ["525.1", "2428d", "525S", "Alter Postweg120", "MAH", 300, "offen", "WV650"], ["539.2", "2529a", "539S", "Salomon-ldler-Straße 22", "MAH", 150, "offen", "WV650"], ["541.1", "2529a", "541S", "Salomon-ldler-Straße 18", "MAH", 150, "offen", "WV650"], ["555.1", "2530a", "555S", "Josef-Priller-Straße 10", "MAH", 150, "offen", "WV650"], ["555.2", "2530a", "555S", "Josef-Priller-Straße10", "MAH", 100, "offen", "WV650"], ["561.1", "2329c", "561EZ", "Alter Postweg93 (Heizwerk Süd Südeinspeisung1)", "MAH", 350, "offen", "WV650"], ["561.2", "2329c", "561EZ", "Alter Postweg93 (Heizwerk Süd Südeinspeisung2)", "MAH", 200, "offen", "WV650"], ["561.3", "2329c", "561EZ", "Alter Postweg93 (Heizwerk Süd Betriebswarme)", "MAH", 100, "offen", "WV650"], ["564.1", "1629a", "564G", "Metzplatz (im Keller des Gebäudes nordliche Seite)", "MAH", 65, "offen", "WV650"], ["591.2", "2129d", "591S", "Johann-Georg-Halske-Straße", "MAH", 200, "offen", "WV650"], ["603.4", "1623b", "603EZ", "Flandern Straße 10 (Heizwerk West Primareinspeisung)", "MAS", 350, "offen", "WV650"], ["735.1", "1828d", "735S", "Theodeor-Heuss-Platz MAH2", "?", 200, "offen", "WV650"], ["750.1", "0935a", "750S", "Pottmeser Straße (Richtung GT-Ost)", "MAH", 350, "offen", "WV650"], ["750.3", "0935a", "750S", "Pottmeser Straße (Richtung Pottmeser Straße)", "MAH", 350, "offen", "WV650"], ["769.1", "1831b", "769E", "Schafflerbach Straße", "AH", 65, "zu", "WV650"], ["773.1", "1034a", "773S", "Affinger Straße (Richtung Biomasse-HKW)", "MAH", 350, "offen", "WV650"], ["773.2", "1034a", "773S", "Affinger Straße (Richtung Affinger Straße)", "MAH", 300, "offen", "WV650"], ["773.3", "1034a", "773S", "Affinger Straße (Richtung Neuburger Straße)", "MAH", 350, "offen", "WV650"], ["777.1", "1132a", "777S", "Kurt-Schumacher-Straße (Richtung Pottmeser Straße)", "MAH", 350, "offen", "WV650"], ["777.2", "1132a", "777S", "Kurt-Schumacher-Straße (Richtung Neuburger Straße)", "MAH", 350, "offen", "WV650"], ["777.3", "1132a", "777S", "Kurt-Schumacher-Straße (Richtung Medienzentrum)", "MAH", 250, "offen", "WV650"], ["783.1", "1832b", "783S", "Reichenberger Straße", "MAH", 250, "offen", "WV650"], ["783.2", "1832b", "783S", "Reichenberger Straße-Richtung KUKA", "MAH", 250, "offen", "WV650"], ["783.3", "1832b", "783S", "Reichenberger Straße-Richtung Schwabencenter", "MAH", 200, "offen", "WV650"], ["784.1", "0835a", "784EZ", "Beim Grenzgraben 20 (Biomasse-HKW)", "MAH", 250, "offen", "WV650"], ["785.1", "1728d", "785G", "Maximilian Straße46 (Schaezler Palais)", "MAH", 125, "offen", "WV650"], ["809.1", "0934a", "809S", "Am Mittleren Moos (Richtung GT-Ost)", "MAH", 400, "offen", "WV650"], ["809.2", "0934a", "809S", "Am Mittleren Moos/Ulstett Straße (SL5 Richtung Westen)", "MAH", 400, "offen", "WV650"], ["810.1", "0834c", "810EZ", "Am Mitleren Moos 60 (AVA-Ubemahmestation)", "MAH", 350, "offen", "WV650"], ["818.1", "1733b", "818S", "Proviantbach Straße Richtung Westen", "MAH", 125, "offen", "WV650"], ["818.2", "1733b", "818S", "Proviantbach Straße Richtung Osten", "MAH", 250, "offen", "WV650"], ["818.3", "1733b", "818S", "Proviantbach Straße Richtung Suiden", "MAH", 250, "offen", "WV650"], ["832.1", "1131b", "832S", "Neuburger Straße (Richtung Steinerne Furt)", "MAH", 350, "offen", "WV650"], ["832.2", "1131b", "832S", "Neuburger Straße (Richtung Neuburger Straße Süd)", "MAH", 350, "offen", "WV650"], ["832.3", "1131b", "832S", "Neuburger Straße (Richtung Hans-Böckler-Straße)", "MAH", 350, "offen", "WV650"], ["833.1", "1131a", "833S", "Hans-Böckler-Straße (Richtung GT-Ost)", "MAH", 400, "offen", "WV650"], ["833.3", "1131a", "833S", "Hans-Böckler-Straße (Richtung MAN)", "MAH", 400, "offen", "WV650"], ["834.1", "2628b", "834S", "Am Technologiezentrum", "MAH", 200, "offen", "WV650"], ["860.1", "2229c", "860E", "Alter Postweg (nordl.Parkplatz RWS)", "AH", 200, "offen", "WV650"], ["861.1", "1528b", "861G", "Frauentor Straße 26", "MAH", 100, "offen", "WV650"], ["864.1", "2429c", "864G", "Alter Postweg115", "MAH", 200, "offen", "WV650"], ["887.1", "1428a", "887S", "Thomm Straße/Trennung Am Pfannenstiel (Nordspange)", "MAH", 300, "offen", "WV650"], ["888.1", "1334d", "888S", "Südtiroler Straße/Kalterer Straße", "MAH", 350, "offen", "WV650"], ["891.1", "1731b", "891S", "Provino Straße", "MAH", 250, "offen", "WV650"], ["894.1", "2028c", "894S", "Windprecht Straßesidlich Scherlin Straße", "MAH", 250, "offen", "WV650"], ["895.1", "1827b", "895S", "Ladehofe", "MAH", 300, "offen", "WV650"], ["995.1", "1634a", "995G", "Zugspitz Straße/KUKA", "MAH", 250, "offen", "WV650"], ["603.1", "1623b", "603EZ", "Flandem Straße10 (Heizwerk West SEK1)", "MAS", 300, "offen", "WV660"], ["603.2", "1623b", "603EZ", "Flandem Straße10 (Heizwerk West SEK2)", "MAS", 300, "offen", "WV660"], ["603.3", "1623b", "603EZ", "Flandem Straße10 (Heizwerk West SEK3)", "MAS", 300, "offen", "WV660"], ["605.1", "1321b", "605S", "Dr.-Dürrwanger-Straße-Richtung SEK1", "MAH", 250, "offen", "WV660"], ["605.2", "1321b", "605S", "Dr.-Dürrwanger-Straße-Richtung SEK2", "MAH", 200, "offen", "WV660"], ["613.1", "1323d", "613S", "Ulmer Straße", "MAH", 250, "offen", "WV660"], ["622.1", "1523b", "622S", "Am Exerzierplatz", "AH", 200, "zu", "WV660"], ["622.2", "1523b", "622S", "Am Exerzierplatz", "AS", 200, "zu", "WV660"], ["631.1", "1622b", "631S", "Columbus Straße7-Richtung Heizwerk West", "AS", 250, "offen", "WV660"], ["631.2", "1622b", "631S", "Columbus Straße7-Richtung Sheridanpark", "MAH", 250, "offen", "WV660"], ["631.3", "1622b", "631S", "Columbus Straße7-Richtung Zentralklinikum", "AS", 250, "offen", "WV660"], ["631.4", "1622b", "631S", "Columbus Straße7-Richtung Reesepark", "AS", 250, "offen", "WV660"], ["650.1", "1522b", "650S", "Carl-Schurz-Straße 16", "MAH", 200, "offen", "WV660"], ["734.1", "1922c", "734E", "General-Cramer-Weg", "AH", 150, "offen", "WV660"], ["797.1", "1822d", "797S", "Grasiger Weg-Richtung Heizwerk West", "MAH", 200, "offen", "WV660"], ["797.2", "1822d", "797S", "Grasiger Weg-Richtung Sheridanpark", "MAH", 150, "offen", "WV660"], ["797.3", "1822d", "797S", "Grasiger Weg-Richtung Schacht850", "MAH", 200, "offen", "WV660"], ["816.1", "1223c", "816S", "Bertha-von-Suttner-Straße", "MAH", 200, "offen", "WV660"], ["850.1", "1822c", "850S", "Siegfried-Aufhäuser-Straße", "MAH", 200, "offen", "WV660"], ["850.3", "1822c", "850S", "Siegfried-Aufhäuser-Straße", "MAH", 150, "offen", "WV660"], ["870.1", "1219a", "870S", "Geschwister-Schönert-Straße-Richtung Neusäß", "MAH", 200, "offen", "WV660"], ["870.2", "1219a", "870S", "Geschwister-Schönert-Straße-Richtung UNI-MED", "MAH", 200, "offen", "WV660"]]

armaturen = pd.DataFrame(ARMATUREN_ROWS, columns=["armatur_nr","schaltplan","anlage","strasse","typ","dn_mm","normalstatus","planblatt"])
SEPARIERUNG_650 = ["141.1","300.1","112.1","080.2","080.3","203.1","166.1","735.1","253.1","785.1","325.1","187.1","832.3","777.3","809.1"]
armaturen["separierung_schliesst"] = armaturen.armatur_nr.isin(SEPARIERUNG_650)
armaturen["man_relevant"] = armaturen.armatur_nr.isin(["342.1","342.2","342.3","833.1","833.3","809.2","001.2"])
armaturen.to_csv(RESULTS/"plan_armaturen.csv", index=False)
print(len(armaturen), "Armaturen | Normalstatus:", armaturen.normalstatus.value_counts().to_dict())
print("Normal ZU (im Plan als geschlossen geführt – bitte prüfen):", armaturen.loc[armaturen.normalstatus=='zu',['armatur_nr','strasse','dn_mm']].values.tolist())
print("\nMAN-relevante Armaturen:"); display(armaturen.loc[armaturen.man_relevant,["armatur_nr","strasse","typ","dn_mm","normalstatus"]])
print("Armaturen der Separierung (WV650, Befehl 'Separierung' schließt diese Notabsperrgruppen):")
display(armaturen.loc[armaturen.separierung_schliesst,["armatur_nr","strasse","dn_mm","normalstatus"]])
# Vollständigkeit der Separierungsliste
fehlend = sorted(set(SEPARIERUNG_650) - set(armaturen.armatur_nr)); print("Separierungs-Armaturen nicht in der Tabelle:", fehlend or "keine")

### 9.4 Transportgrenzen und Plausibilität der Trassen-Nennweiten
Die im Plan genannten gesicherten Transportkapazitäten im separierten Betrieb (SL5 750, SL6 500, SL7 300 t/h) werden mit den in der Armaturenliste abgelesenen Nennweiten der jeweiligen Trasse auf Strömungsgeschwindigkeit geprüft.

In [ ]:
RHO_HOT = 960.0
def v_ms(t_h, dn_mm): return (t_h*1000/3600)/(RHO_HOT*np.pi*(dn_mm/1000)**2/4)
def q_mw(t_h, dT=39.0): return t_h*1000/3600*4.19*dT/1000
tg = pd.DataFrame([("SL5",750,400,"Armaturen 001.2, 342.1/.3, 809.x: DN400"),("SL6",500,350,"Armaturen 338.2, 832.x, 750.x: DN350"),("SL7",300,250,"Armaturen 818.2/.3, 995.1: DN250 (888.1: DN350)")],
                  columns=["trasse","gesicherte_Kapazitaet_t/h","DN_Plan_mm","Fundstelle"])
tg["v_bei_Grenze_m/s"] = [v_ms(t,d) for t,d in zip(tg["gesicherte_Kapazitaet_t/h"], tg.DN_Plan_mm)]
tg["Q_bei_dT39K_MW"] = [q_mw(t) for t in tg["gesicherte_Kapazitaet_t/h"]]
display(tg.round(2))
print(f"Summe SL5+SL6+SL7: {tg['gesicherte_Kapazitaet_t/h'].sum()} t/h ≙ {tg.Q_bei_dT39K_MW.sum():.0f} MW bei ΔT=39 K; YAML-Trasse hkw_to_ost: DN600 (Sammelleitung) mit v=3 m/s ≙ 135 MW")
assert 1.0 < tg["v_bei_Grenze_m/s"].min() and tg["v_bei_Grenze_m/s"].max() < 2.5
print("→ Die gesicherten Kapazitäten entsprechen Geschwindigkeiten von etwa 1,5–1,8 m/s in den abgelesenen Nennweiten: die Zuordnung Trasse ↔ DN ist plausibel.")

### 9.5 Vergleich: YAML-Topologie gegen Pläne

In [ ]:
ycfg = yaml.safe_load((ROOT/"configs/paper_2/Stadtbach_topo.yaml").read_text(encoding="utf-8"))
yn, yp = ycfg["network"]["nodes"], ycfg["network"]["pipes"]
E, N = len(yp), len(yn); comps = 1
print(f"YAML: {N} Knoten, {E} Leitungen, unabhängige Schleifen = E − N + Komponenten = {E-N+comps} (0 = Baum)")
vgl = pd.DataFrame([
 ("Netzstruktur","reiner Baum, 1 bidirektionale Trasse","Verbundnetz: 4 Stammleitungen SL1–4 ab HKW, 3 Ost-Trassen SL5–7, Vermaschung über Notabsperr-Armaturen","hoch"),
 ("Ost-Anbindung","1 Sammelleitung hkw_to_ost (DN600, 5,93 km) bis zu einem 'j_ost'","3 getrennte Trassen SL5/SL6/SL7 (DN400/350/250) mit Kopplungen bei 832/777/809 und Einbindung bei 001.2, 338, 818","hoch"),
 ("MAN","Verbraucher j_man, DN400, 850 m ab HKW, direkt am HKW","32-MW-Einzelkunde am SL5-Ende, Armaturen 342.1–3 (DN400/300/400), ~0,3–0,4 km vor Einbindung 001.2 in SL1","hoch"),
 ("Westnetz","direkte Rohrverbindung HKW→PSW→HWW→Kunden (ein Druckniveau)","PN16-Netz, hydraulisch über Wärmeübertrager von PN25 getrennt (Pfersee/PSW, Kapellenschule); Versorgung über SEK1–SEK3 aus HWW","hoch"),
 ("Druckniveau","VL-Sollwert 16 bar am HKW","Ruhedruckbalken 8,4 bar Ü; Pumpen im VL (HKW/PSW/PSS) bzw. im RL (Ost-Erzeuger, HWS, HWW)","hoch"),
 ("Höhen","keine","Bezugshöhen der Erzeuger/Pumpstationen 466,7 … 490,5 m NN (Δ ≈ 24 m ≈ 2,4 bar)","hoch"),
 ("PN-Zonen","keine","PN16 (West/Nordwest), PN25 (Verbund)","hoch"),
 ("Verbrauchsstationen","24 einzelne Stichleitungen (DN125–400) je Station","Sektionen mit 1.504 Kundenanlagen; Stationen = Messstellen an Sektionsleitungen","mittel"),
 ("Erzeugerleistungen","127/35/14,5/45/32/104 MW","identisch (ÜZeit-Werte, WV640); HWW gesamt 104 = 64 Prim + 40 sek.","hoch"),
], columns=["thema","YAML","Plan","Sicherheit_Befund"])
with pd.option_context("display.max_colwidth",95,"display.width",250): display(vgl)
vgl.to_csv(RESULTS/"vergleich_yaml_plan.csv", index=False)

## 10. Aufbau des kanonischen Netzmodells (Sektionsmodell v0.1)
**Auflösung:** Ein Straßen- und Rohrabschnittsmodell ist aus den vorhandenen Rasterplänen nicht ableitbar (keine Rohrlängen/-durchmesser je Abschnitt). Es wird ein **Sektionsmodell** mit Vorlauf- und Rücklaufnetz aufgebaut, das die reale Vermaschung (SL1–SL7, Koppelstellen, Ost-Trassen) abbildet. Rohrlängen sind nur für wenige Trassen belastbar; die übrigen Werte sind als *Ersatzleitungen* gekennzeichnet und werden in Kapitel 13 gegen Messdaten kalibriert.

**Modellierungsentscheidungen (Arbeitsannahmen, bitte prüfen):**
1. **MAN-Anschlusspunkt:** Knoten `MAN` auf der Trasse SL5 zwischen Koppelstelle `K001` (001.2) und `SL5M` (Hans-Böckler-Straße). Der 32-MW-Kunde ist Verbraucher an diesem Knoten; der Speicher wird als zusätzliche Anschlussstelle **am selben Knoten** modelliert (Kapitel 15).
2. **Westnetz (PN16)** wird hydraulisch als *getrennt* behandelt. Die Kopplung zum Verbund erfolgt an `HWW` über Wärmeübertrager (nur thermische Kopplung, Massenströme sind getrennt).
3. **Vermaschung:** Verbindungen zwischen den Sektionen sind Kanten mit Normalstatus *offen*; die Separierung schließt sie.
4. **Höhen:** nur für Erzeuger und Pumpstationen aus WV640 bekannt; für alle übrigen Knoten fehlt die Höhe (`z_NN_m = NaN`) → in Kapitel 13 als Kalibrierkonstante (Höhen-Offset) je Sektion.

In [ ]:
# ------------------------------- Knoten -------------------------------
def _n(i,name,typ,sek,netz,z=np.nan,p_MW=np.nan,x=0,y=0,quelle="",sicher="mittel"):
    return dict(node_id=i,name=name,typ=typ,sektion=sek,netz=netz,PN=(16 if netz=="West" else 25),z_NN_m=z,p_anschluss_MW=p_MW,x_km=x,y_km=y,quelle=quelle,sicherheit=sicher)
knoten = pd.DataFrame([
 _n("HKW","Heizkraftwerk Sammelschiene","erzeuger","-","Mitte",473.77,0,0.0,0.0,"WV640","hoch"),
 _n("K001","Koppelstelle SL1/SL5 (Thommstr., Arm. 001.2)","koppelstelle","SL1","Mitte",np.nan,0,-0.6,0.9,"WV650","mittel"),
 _n("MAN","MAN (Stadtbachstr., Arm. 342.1-3) – Speicher-Anschluss","verbraucher","SL1","Mitte",np.nan,32.0,-0.2,1.4,"WV650 Tabelle SL5-West mit MAN","hoch"),
 _n("SL5M","SL5-Mitte (Hans-Böckler-Str., Arm. 833)","verbraucher","SL1","Ost",np.nan,2.4,0.9,2.5,"WV650","mittel"),
 _n("KNEU","Koppelstelle SL5/SL6 (Neuburger Str., Arm. 832/777)","koppelstelle","SL1/SL3","Ost",np.nan,0,1.7,3.1,"WV650","mittel"),
 _n("SL5O","SL5-Ost (Am Mittleren Moos, Arm. 809.2)","verbraucher","SL1","Ost",np.nan,8.2,3.2,3.7,"WV650","mittel"),
 _n("AVA","AVA-Übernahme","erzeuger","-","Ost",466.69,0,4.2,4.1,"WV640","hoch"),
 _n("GT","GT-Ost","erzeuger","-","Ost",467.98,0,4.6,3.8,"WV640","hoch"),
 _n("BM","Biomasse-HKW","erzeuger","-","Ost",467.08,0,4.9,4.4,"WV640","hoch"),
 _n("SL6S","SL6-Gebiet (Ost+Süd)","verbraucher","SL3","Ost",np.nan,9.4,2.6,1.9,"WV650","mittel"),
 _n("K338","Koppelstelle SL6/SL3 (Lechhauser Str., Arm. 338)","koppelstelle","SL3","Ost",np.nan,0,1.3,0.4,"WV650","mittel"),
 _n("SL7G","SL7-Gebiet (West+Mitte+Ost)","verbraucher","SL3","Ost",np.nan,12.4,4.2,1.0,"WV650","mittel"),
 _n("K818","Koppelstelle SL7/SL3 (Proviantbachstr., Arm. 818)","koppelstelle","SL3","Ost",np.nan,0,2.3,-0.4,"WV650","mittel"),
 _n("SL1G","SL1-Gebiet (West/Mitte/Nord/Ost) ohne SL5","verbraucher","SL1","Mitte",np.nan,6.2,-1.5,0.4,"WV650","mittel"),
 _n("SEC2","Sektion SL2 (DOM + CITY)","verbraucher","SL2","Mitte",np.nan,150.7,-0.7,-0.3,"WV650","mittel"),
 _n("SEC3","Sektion SL3-STAWA","verbraucher","SL3","Mitte",np.nan,58.8,0.9,-0.2,"WV650","mittel"),
 _n("SEC4","Sektion SL4 (SL4 + ULRICH)","verbraucher","SL4","Süd",np.nan,38.6,0.3,-1.2,"WV650","mittel"),
 _n("PSS","Pumpstation Süd","pumpstation","SL4","Süd",481.45,0,0.5,-3.2,"WV640","hoch"),
 _n("SUED","Sektion SÜD","verbraucher","SL4","Süd",np.nan,73.3,0.8,-3.8,"WV650","mittel"),
 _n("HWS","Heizwerk Süd","erzeuger","SL4","Süd",490.51,0,0.6,-4.5,"WV640","hoch"),
 _n("PSW","Pumpstation West","pumpstation","SL1","Mitte",471.55,0,-2.6,-0.6,"WV640","hoch"),
 _n("HWW","Heizwerk West (Primärseite / WÜ)","erzeuger","-","Mitte",477.99,0,-4.0,-0.3,"WV640","hoch"),
 _n("WEST","Westnetz PN16 (SEK1–SEK3, sekundär)","verbraucher","SEK1-3","West",np.nan,75.8,-4.8,0.8,"WV660","mittel"),
])
knoten.to_csv(RESULTS/"modell_knoten.csv", index=False)
print(len(knoten), "Knoten | Typen:", knoten.typ.value_counts().to_dict())
print(f"Σ Anschlussleistung Knoten = {knoten.p_anschluss_MW.sum():.1f} MW (Plan Verbund 392,0 + West 75,8 = 467,8)")

In [ ]:
# ------------------------------- Leitungen / Kanten -------------------------------
# laenge_quelle: Y = YAML-Wert, P = YAML-Summe nach Plan-Trassenanteil aufgeteilt, S = Schätzung (Sicherheit niedrig)
def _e(i,a,b,art,dn,L,lq,sich,stat,arm,qmax=np.nan,bidir=True):
    return dict(edge_id=i,von=a,nach=b,art=art,dn_mm=dn,laenge_m=L,laenge_quelle=lq,sicherheit=sich,normalstatus=stat,armaturen=arm,max_t_h_separiert=qmax,bidirektional=bidir)
# SL5-Gesamtlänge 5.930 m (YAML, HKW->Ost) nach Pixelanteilen der Plan-Trasse verteilt (HKW-K001 14,4 %, K001-MAN 6,7 %, MAN-SL5M 38,5 %, SL5M-KNEU 4,4 %, KNEU-AVA 36,1 %)
kanten = pd.DataFrame([
 _e("SL1a","HKW","K001","Stammleitung SL1",400,854,"P","niedrig","offen","01, 001.2"),
 _e("SL5a","K001","MAN","Trasse SL5",400,394,"P","niedrig","offen","342.1, 342.3"),
 _e("SL5b","MAN","SL5M","Trasse SL5",400,2285,"P","niedrig","offen","833.1, 833.3",750),
 _e("SL5c","SL5M","KNEU","Trasse SL5",400,258,"P","niedrig","offen","832.x",750),
 _e("SL5d","KNEU","SL5O","Trasse SL5",400,1800,"P","niedrig","offen","809.2",750),
 _e("SL5e","SL5O","AVA","Erzeugeranschluss",350,870,"Y","mittel","offen","810.1, 809.x",750),
 _e("SL5f","SL5O","GT","Erzeugeranschluss",350,60,"Y","mittel","offen","809.1, 515.1",750),
 _e("SL5g","SL5O","BM","Erzeugeranschluss",250,60,"Y","mittel","offen","784.1, 773.1",750),
 _e("SL6a","KNEU","SL6S","Trasse SL6",350,1500,"S","niedrig","offen","777.x, 832.1",500),
 _e("SL6b","SL6S","K338","Trasse SL6",350,1500,"S","niedrig","offen","338.2",500),
 _e("SL6c","KNEU","GT","Trasse SL6 (Ost)",350,3000,"S","niedrig","offen","773.x, 750.x",500),
 _e("SL7a","K818","SL7G","Trasse SL7",250,2500,"S","niedrig","offen","818.2, 818.3",300),
 _e("SL7b","SL7G","GT","Trasse SL7 (Ost)",250,3500,"S","niedrig","offen","888.1, 995.1",300),
 _e("SL2","HKW","SEC2","Stammleitung SL2",350,600,"S","niedrig","offen","02"),
 _e("SL3","HKW","SEC3","Stammleitung SL3",350,800,"S","niedrig","offen","03, 050.x"),
 _e("SL4a","HKW","SEC4","Stammleitung SL4",400,1200,"S","niedrig","offen","04, 294.1"),
 _e("SL4b","SEC4","PSS","Stammleitung SL4",400,3270,"P","niedrig","offen","297.1"),
 _e("SL4c","PSS","SUED","Stammleitung SL4",400,610,"Y","mittel","offen","502.x"),
 _e("SL4d","SUED","HWS","Erzeugeranschluss",350,900,"S","niedrig","offen","561.1/.2"),
 _e("SL1b","K001","SL1G","Stammleitung SL1",350,1400,"S","niedrig","offen","001.2, 007.1"),
 _e("SL1c","SL1G","PSW","Stammleitung SL1",350,1400,"S","niedrig","offen","080.x, 346.1"),
 _e("W1","PSW","HWW","Prim-Leitung West",350,960,"Y","mittel","offen","346.1, 603.4"),
 # Verbundkanten (Vermaschung): im Normalbetrieb offen, bei Separierung geschlossen
 _e("M_SL1_SL2","SL1G","SEC2","Verbund SL1↔SL2",350,700,"S","niedrig","offen","080.2, 080.3, 141.1, 300.1, 112.1"),
 _e("M_SL2_SL3","SEC2","SEC3","Verbund SL2↔SL3",200,600,"S","niedrig","offen","203.1, 214.1, 050.1/.2"),
 _e("M_SL2_SL4","SEC2","SEC4","Verbund SL2↔SL4",200,800,"S","niedrig","offen","735.1, 785.1, 325.1, 253.1, 187.1"),
 _e("M_SL3_SL4","SEC3","SEC4","Verbund SL3↔SL4",250,700,"S","niedrig","offen","166.1"),
 _e("K_SL6_SL3","K338","SEC3","Landung SL6 in SL3",350,300,"S","niedrig","offen","338.1-3"),
 _e("K_SL7_SL3","K818","SEC3","Landung SL7 in SL3",250,600,"S","niedrig","offen","818.1-3, 783.x"),
], columns=None)
kanten["laenge_km"] = kanten.laenge_m/1000
kanten.to_csv(RESULTS/"modell_kanten.csv", index=False)
print(len(kanten), "Kanten; Längenquelle:", kanten.laenge_quelle.value_counts().to_dict(), "| Σ Länge:", round(kanten.laenge_m.sum()/1000,1), "km")

# --------- Topologie-Kennzahlen (scipy statt networkx, keine zusätzliche Abhängigkeit)
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
idx = {n:i for i,n in enumerate(knoten.node_id)}
A = coo_matrix((np.ones(len(kanten)), ([idx[a] for a in kanten.von],[idx[b] for b in kanten.nach])), shape=(len(idx),)*2)
ncomp, lab = connected_components(A, directed=False)
print(f"Zusammenhangskomponenten: {ncomp} (Knoten je Komponente: {np.bincount(lab).tolist()})")
print(f"Unabhängige Schleifen (E − N + K): {len(kanten)-len(knoten)+ncomp}  → das Modell ist vermascht (YAML: 0)")
print("Die 2. Komponente ist der Knoten WEST (PN16): bewusst nur thermisch über Wärmeübertrager an HWW gekoppelt, keine hydraulische Kante.")
assert all(k in idx for k in list(kanten.von)+list(kanten.nach))

In [ ]:
# --------- Schaubild des Sektionsmodells
fig, ax = plt.subplots(figsize=(12,8.5))
pos = knoten.set_index("node_id")[["x_km","y_km"]]
farbe = {"erzeuger":COL["warn"],"pumpstation":COL["ok"],"verbraucher":COL["mess"],"koppelstelle":COL["grau"]}
for _,e in kanten.iterrows():
    a,b = pos.loc[e.von], pos.loc[e.nach]
    verbund = e.edge_id.startswith("M_")
    ax.plot([a.x_km,b.x_km],[a.y_km,b.y_km], color=COL["vl"] if not verbund else COL["rl"], lw=1+e.dn_mm/150, ls="--" if verbund else "-", alpha=.85, zorder=1)
for _,n in knoten.iterrows():
    s = 120+ (np.sqrt(n.p_anschluss_MW)*25 if n.p_anschluss_MW==n.p_anschluss_MW else 0)
    ax.scatter(n.x_km,n.y_km,s=s,color=farbe[n.typ],edgecolor="k",zorder=3, marker="*" if n.node_id=="MAN" else "o")
    ax.annotate(n.node_id, (n.x_km,n.y_km), xytext=(6,6), textcoords="offset points", fontsize=9, fontweight="bold" if n.node_id=="MAN" else None)
ax.set(xlabel="schematisch Ost →  [km, nicht maßstäblich]", ylabel="schematisch Nord →  [km]", title="Sektionsmodell v0.1 (Vorlauf/Rücklauf je Kante); gestrichelt: Verbundkanten (Normalstatus offen); ★ MAN/Speicher")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([],[],marker="o",ls="",color=c,label=l,markeredgecolor="k") for l,c in farbe.items()]+[Line2D([],[],color=COL["rl"],ls="--",label="Verbundkante")], loc="lower right")
ax.set_aspect("equal"); fig.tight_layout(); fig.savefig(FIG_DIR/"10_sektionsmodell.png"); plt.show()

### 10.2 Zuordnung der Messstellen zum Modell
Zuordnung der 34 ACRON-Stationen (Ordner) zu Modellknoten. **Sicherheit:** *hoch* = Anlage im Plan eindeutig identifiziert; *mittel* = Zuordnung über Straßenname/Sektion, plausibel; *niedrig* = Sektion nur vermutet. Unsichere Zuordnungen fließen als Sensitivität in Kapitel 13/21 ein.

In [ ]:
zuord = {
 "HKW":("HKW","hoch"),"AVA":("AVA","hoch"),"GT-Ost":("GT","hoch"),"BMHKW":("BM","hoch"),"MAN":("MAN","hoch"),
 "PSS":("PSS","hoch"),"PSW":("PSW","hoch"),"Heizwerk_Sued":("HWS","hoch"),"HWW_prim_Einsp":("HWW","hoch"),"HWW_Kessel":("HWW","mittel"),"HWW_sek_Aussp":("WEST","mittel"),
 "Hans-Boeckler-Str":("SL5M","hoch"),"Am_Mittleren_Moos":("SL5O","hoch"),"Kurt-Schumacher-Str":("SL5O","mittel"),
 "Lechhauser_Str":("SEC3","mittel"),"Don_Bosco":("SEC3","hoch"),"KUKA":("SL7G","mittel"),
 "Hoher_Weg":("SEC2","hoch"),"Hunoldsgraben":("SEC2","mittel"),"Kreissparkasse":("SEC2","mittel"),"Fuggerstr":("SEC2","mittel"),"Schlettererstr":("SL1G","niedrig"),
 "Theodor-Heuss-Platz":("SEC4","hoch"),"UNI":("SUED","hoch"),"SIGMA_Technopark":("SUED","hoch"),"Fraunhofer":("SUED","niedrig"),"Beethovenpark":("SUED","niedrig"),
 "Klinikum":("WEST","hoch"),"Josefinum":("WEST","mittel"),"Lise-Meitner-Str":("WEST","niedrig"),"August-Wessels-Str":("WEST","niedrig"),
 "Hooverstr":("WEST","mittel"),"Grasiger_Weg":("WEST","mittel"),"Siegfried_Aufhaeuser-Str":("WEST","mittel"),
}
stationen = sorted(set(meta.ordner))
fehl = [s for s in stationen if s not in zuord and s.replace("-","_") not in zuord and s.replace("_","-") not in zuord]
zuord["Siegfried-Aufhaeuser-Str"] = zuord["Siegfried_Aufhaeuser-Str"]
messz = []
for s in stationen:
    k, sich = zuord.get(s, (None,"offen"))
    cols = [c for c in acron.columns if meta.ordner[c]==s]
    messz.append(dict(station=s, modellknoten=k, sicherheit=sich, kanaele=", ".join(sorted({meta.kanal[c] for c in cols})),
                      n_signale=len(cols), abdeckung_median_pct=round(float(qual.loc[cols,"abdeckung_pct"].median()),1)))
messz = pd.DataFrame(messz)
messz.to_csv(RESULTS/"messstellenzuordnung.csv", index=False)
print("Nicht zugeordnete Stationen:", messz[messz.modellknoten.isna()].station.tolist() or "keine")
print("Sicherheit:", messz.sicherheit.value_counts().to_dict())
with pd.option_context("display.width",200): display(messz)

### 10.3 Kritische Plausibilitätsprüfung des Modells gegen die Messdaten
Ziel: prüfen, ob die aus den Plänen abgelesene Struktur mit dem gemessenen Betrieb vereinbar ist, **bevor** darauf gerechnet wird. Vier Tests: (a) Flussrichtungen an den HKW-Stammleitungen, (b) hydraulische Druckhöhen der Anlagen, (c) hydraulische Entkopplung des Westnetzes, (d) Druckverlust-Ansatz Δp = c₀ + k·Q|Q| je Stammleitung und Vergleich mit den Modell-Ersatzleitungen.

In [ ]:
# (a) Flussrichtungen (+ = aus dem HKW ins Netz)
fr = []
for k in ("SL1","SL2","SL3","SL4"):
    s_ = acron[f"HKW_Durchfluss_{k}"].dropna()
    fr.append((k, round(100*(s_<0).mean()), round(s_.median()), round(s_.min()), round(s_.max())))
fr = pd.DataFrame(fr, columns=["Stammleitung","Anteil_Rückfluss_ins_HKW_%","Median_m3/h","min","max"]); display(fr)
print("→ SL1 und SL3 führen überwiegend Wärme *in* die HKW-Sammelschiene (Ost-Einspeisung über SL5 bzw. SL6/SL7), SL2 und SL4 nur aus dem HKW heraus: passt zur abgelesenen Struktur (SL5→SL1, SL6/SL7→SL3).")

# (b) hydraulische Druckhöhen der Anlagen  h = z + p/(ρ g)
zA = {"HKW":473.77,"AVA":466.69,"GT-Ost":467.98,"BMHKW":467.08,"Heizwerk_Sued":490.51}
hh = pd.DataFrame([(st, z0, acron[f"{st}_Druck_VL"].median(), acron[f"{st}_Druck_RL"].median(),
                    z0+acron[f"{st}_Druck_VL"].median()*1e5/(960*9.81), z0+acron[f"{st}_Druck_RL"].median()*1e5/(960*9.81)) for st,z0 in zA.items()],
                  columns=["Anlage","z_Plan_mNN","p_VL_bar","p_RL_bar","Druckhöhe_VL_mNN","Druckhöhe_RL_mNN"]).round(1)
display(hh)
print("→ HKW-Rücklauf-Druckhöhe ≈ Ruhedruck-Druckhöhe (561,8 m): Höhenbezug und Überdruck-Annahme konsistent. AVA/BM-HKW liegen im Betrieb ~10–15 m über dem HKW (Fluss Ost→West). GT-Ost steht überwiegend still (Median = Netzdruck).")
print("→ Die geodätischen Unterschiede (bis ~24 m ≙ 2,4 bar) sind größer als die Zieldruckdifferenzen (0,7–4 bar): Höhen sind für jeden Knoten zu kalibrieren.")

# (c) Entkopplung Westnetz: Partialkorrelation der Rücklaufdrücke nach Herausrechnen von Außentemperatur und Last
Xc = pd.concat([Ta, acron["HKW_Durchfluss_gesamt"]], axis=1).dropna()
def resid(s_):
    d = pd.concat([s_, Xc], axis=1).dropna(); X = np.c_[np.ones(len(d)), d.iloc[:,1:].values]
    return d.iloc[:,0] - X @ np.linalg.lstsq(X, d.iloc[:,0], rcond=None)[0]
Rr = pd.DataFrame({k: resid(acron[v]) for k,v in {"HKW_RL":"HKW_Druck_RL","HWW_prim_RL":"HWW_Druck_RL_Primeinsp","HWW_sek_RL":"HWW_Druck_RL","Klinikum_RL":"Klinikum_Druck_RL","Grasiger_RL":"Grasiger_Weg_Druck_RL","Sued_RL":"Heizwerk_Sued_Druck_RL"}.items()})
print("Streuung (σ) des Rücklaufdrucks [bar]: HWW-sekundär", round(acron.HWW_Druck_RL.std(),2), "| HKW", round(acron.HKW_Druck_RL.std(),2), "| HWW-primär", round(acron.HWW_Druck_RL_Primeinsp.std(),2))
display(Rr.corr().round(2).loc[["Klinikum_RL","Grasiger_RL","HWW_sek_RL"],["HKW_RL","HWW_prim_RL","Sued_RL","HWW_sek_RL","Klinikum_RL","Grasiger_RL"]])
print("→ Der Westnetz-Rücklauf ist praktisch konstant (σ≈0,05 bar, eigene Druckhaltung ~6,6 bar) und nach Herausrechnen der gemeinsamen Last unkorreliert mit dem Verbund-Rücklauf (|r|≤0,06). Die West-Stationen korrelieren untereinander (0,5). Das stützt eine hydraulische Entkopplung (Wärmeübertrager).")

In [ ]:
# (d) Druckverlust-Ansatz je Stammleitung: Δp = c0 + k·Q|Q|   (c0 ≙ geodätischer Höhen-/Sensor-Offset, Q in m3/h)
def fit_dp(dp, Q):
    d = pd.concat([dp, Q], axis=1, keys=["dp","Q"]).dropna(); X = np.c_[np.ones(len(d)), d.Q*d.Q.abs()/1e4]
    b = np.linalg.lstsq(X, d.dp, rcond=None)[0]; r2 = 1-((d.dp-X@b)**2).sum()/((d.dp-d.dp.mean())**2).sum(); return b[0], b[1], r2, len(d)
def coef_bar_per_m(dn_mm, lam=0.02, rho=960.0):
    D = dn_mm/1000; A = np.pi*D**2/4; return lam/D*rho/2/(3600*A)**2/1e5      # bar je m Rohr je (m3/h)^2 (einfache Leitung)
tests = [("SL1 → MAN (VL)", acron.MAN_Druck_VL_Station-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL1, 400, 854+394),
         ("SL2 → Hoher Weg", acron.Hoher_Weg_Druck_VL-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL2, 350, 600),
         ("SL3 → Lechhauser Str.", acron.Lechhauser_Str_Druck_VL-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL3, 350, 800),
         ("SL4 → Theodor-Heuss-Pl.", acron["Theodor-Heuss-Platz_Druck_VL"]-acron.HKW_Druck_VL, -acron.HKW_Durchfluss_SL4, 400, 1200)]
rows = []
for nm, dp, Q, dn, Lmod in tests:
    c0, k, r2, n = fit_dp(dp, Q); Leq = (k*1e-4)/coef_bar_per_m(dn)
    rows.append((nm, n, round(c0,2), round(k,3), round(r2,2), dn, round(Leq), Lmod))
fitdf = pd.DataFrame(rows, columns=["Pfad","n","c0_bar","k_bar/(1e4 m3/h)²","R²","DN_Stammleitung","L_äquivalent_m","L_Modell_v0.1_m"]); display(fitdf)
fitdf.to_csv(RESULTS/"pruefung_druckverlust_fit.csv", index=False)
print("→ Die aus den Messdaten abgeleiteten äquivalenten Längen (≈1,0–2,6 km) liegen um den Faktor 1,2–3 über den geschätzten Modell-Längen v0.1 (0,6–1,25 km): die Ersatzleitungen sind zu kurz und werden kalibriert. SL2 und SL4 werden gut erklärt (R² 0,5–0,6), SL3 schwach (0,23), SL1→MAN kaum (0,13): der SL1-Fluss enthält Abzweige (SL1-West, PSW), und die Druckdifferenz MAN–HKW liegt nahe der Sensorauflösung (0,1 bar). Die Hydraulik um MAN ist damit aus den Messdaten nur schwach bestimmt.")
print("→ c0 von −0,8 … −1,6 bar ≙ 8–16 m Höhen-/Sensoroffset gegenüber dem HKW: bestätigt die Notwendigkeit der Höhenkalibrierung.")

In [ ]:
# MAN-Last und Erzeugersignale
man = acron[["MAN_Waermeleistung_Station","MAN_Durchfluss_Station"]].dropna()
print(f"MAN gemessen: Q_max {man.MAN_Waermeleistung_Station.max():.1f} MW, 99 %-Quantil Winter {acron.MAN_Waermeleistung_Station[acron.index.month.isin([12,1,2])].quantile(.99):.1f} MW, Q_max/Anschlussleistung = {man.MAN_Waermeleistung_Station.max()/32.0:.0%}; V̇_max {man.MAN_Durchfluss_Station.max():.0f} m3/h")
v342 = man.MAN_Durchfluss_Station.max()/3600/(np.pi*0.3**2/4)
print(f"→ In der DN300-Armatur 342.2 (Abzweig zu MAN) entspricht V̇_max nur {v342:.2f} m/s: der MAN-Kundenanschluss ist hydraulisch schwach belastet; die 32 MW sind Anschlussleistung, nicht Gleichzeitigkeitslast.")
hw = acron.filter(regex="^HWW_(Waermeleistung|Durchfluss)").describe().loc[["count","50%","max"]].round(1).T; display(hw)
print("→ HWW_Waermeleistung_Primeinsp erreicht 95,5 MW, der Plan nennt für die Prim-Einspeisung max. 64 MW (95 ≈ 64 + 31 MW Prim-Ausspeisung): Bedeutung/Vorzeichen des Signals ungeklärt. Für AVA existiert kein Durchflusssignal, für GT-Ost nur 35 % Abdeckung.")

**Ergebnis der Plausibilitätsprüfung**
* *Bestätigt:* Rückflussrichtungen SL1/SL3, Druckhöhen der Ost-Erzeuger, Ruhedruck-Bezug, Entkopplung des Westnetzes, Größenordnung der Rohrwiderstände (Modell-Längen jedoch zu kurz).
* *Zweifelhaft / offen:* Armaturen 141.1 und 187.1 (im Plan „ZU“ und zugleich in der Separierungsliste), MAN als T-Abzweig DN300 statt Streckenknoten, Höhen/Sensorhöhen, Bedeutung einzelner Erzeugersignale, fehlende Schaltzustands-Historie, schwache Identifizierbarkeit der Strecke HKW–MAN.

### 10.4 Verbleibende Aggregationen und Einschränkungen
| Aggregation | hydraulische Aussage, die dadurch eingeschränkt ist |
|---|---|
| Sektionen (SL2, SL3, SL4, SÜD, West) als Einzelknoten mit Ersatzleitung | Druckverteilung *innerhalb* der Sektion, Verhalten einzelner Straßen/Stationen; Kundendifferenzdruck nur aus Messwerten, nicht aus Rohrnetz |
| Trassenlängen SL5–SL7 aus YAML-Summen bzw. geschätzt | absolute Druckverluste in den Ost-Trassen (Kalibrierung in Kap. 13 erforderlich) |
| Höhen nur an Erzeugern/Pumpstationen | geodätischer Anteil der Druckdifferenzen zwischen Knoten; wird als Offset kalibriert |
| Westnetz als ein Lastknoten hinter WÜ | Druckverhältnisse im Westnetz; SEK-Separierung (WV660) in dieser Etappe nicht abgebildet |
| Verbundkanten je Sektionspaar gebündelt | Einzelne Armaturenschaltungen (z. B. 080.2 vs. 080.3) werden gemeinsam geschaltet |

---
**Ende Etappe 2 – Halt zur Prüfung.** Bitte prüfen (siehe Nachricht im Chat): Knoten/Kanten/Verbundkanten, MAN-Lage, Westnetz-Entkopplung, Armaturen-Normalstatus, Rasterweite der Pläne, Messstellenzuordnung.